# ShiftLens: A Leakage-Controlled, Intervention-Based Diagnostic of Cross-Corpus Shift in Low-Resource Hate-Speech Detection

**Reproducible experiment notebook**

This notebook implements a narrow, identifiable two-factor design for isolating cross-corpus shift in low-resource hate-speech detection:

1. A common Bangla anchor corpus is split once into disjoint train, validation, adaptation, and evaluation groups.
2. **Content shift** is an evaluation-distribution intervention: topic composition is reweighted while the class prior and source remain controlled.
3. **Protocol shift** is a training-label intervention: a pre-specified class-conditional annotation-noise matrix corrupts observed training labels while clean latent labels remain available only for audit/evaluation.
4. Platform and language/register differences are treated as **natural external shifts**, not as synthetically isolated factors. They are evaluated on held-out real corpora and are never included in the synthetic Shapley claim.
5. Exact two-factor Shapley decomposition is computed from a complete 2×2 factorial, with seed-level uncertainty and paired randomization tests.
6. ERM, DANN, CORAL, and a transition-corrected classifier are compared under identical splits, seeds, budgets, and source-validation model selection.

The central claim is deliberately limited:

> Under the stated intervention definitions, the notebook identifies the contributions of content-composition shift and annotation-protocol noise to performance degradation. It does **not** claim that a real-world corpus pair has a causally identified platform, language, or content attribution.

Reporting completing full run results, all failures, and the data-processing/evaluation code is required before any claim in the paper is drawn from this notebook.

## 1. Research question, hypotheses, and claim discipline

### Research question

Can a leakage-controlled intervention protocol separate the performance loss caused by **content-composition shift** from the loss caused by **annotation-protocol noise** in low-resource hate-speech detection, and can standard adaptation/correction methods recover only the component they are designed to address?

### Pre-registered hypotheses

| ID | Hypothesis | Test | Interpretation if supported |
|---|---|---|---|
| H1 | Content intervention causes a reproducible performance drop | Paired seed-level comparison of the 2×2 factorial | Evidence for an intervention-defined content effect |
| H2 | Protocol intervention causes a reproducible performance drop | Paired seed-level comparison of the 2×2 factorial | Evidence for an intervention-defined protocol effect |
| H3 | The two effects are non-additive | Seed-block interaction estimate and randomization interval | The attribution must retain an interaction component |
| H4 | Transition correction preferentially helps protocol-shifted cells | Paired comparison of `PROTOCOL_CORRECTED` and `ERM` | Selectivity evidence, not proof of causal mechanism in real corpora |
| H5 | Synthetic findings transfer descriptively, not causally, to real corpora | External-corpus stress test with signature and OOD reporting | Only a transportability diagnostic; no real Shapley claim |

### Evidence-based novelty and contribution

This work should be positioned as a **design-based attribution benchmark**, not as a first general solution to cross-corpus hate-speech shift. The defensible contribution consists of four linked components:

1. **A leakage-controlled intervention protocol:** content-composition shift is created by class-preserving, without-replacement re-sampling from naturally annotated topics, while annotation-protocol shift is created by a declared class-conditional transition matrix. The two factors are isolated in a balanced 2×2 design with a held-out reserve pool.
2. **An estimand-aware attribution analysis:** exact two-factor Shapley values, interaction terms, paired seed-level tests, and uncertainty intervals quantify what the intervention protocol can identify, rather than assigning unverifiable language/platform percentages to naturally occurring corpora.
3. **Method selectivity tests:** ERM, DANN, CORAL, and transition correction are compared under identical data, seeds, checkpoint rules, and budgets to test whether each method improves the shift component it is theoretically intended to address.
4. **A transportability boundary:** BD-SHS, HateXplain, and BengaliHate are treated as external stress tests with observable signatures and OOD flags. Their results are used to test descriptive consistency, not to claim causal transfer of the synthetic decomposition.

The novelty claim is permitted only if the completed results show: (a) all declared model–seed–cell runs completed, (b) the content intervention changes topic composition while preserving class prior and sample size, (c) the protocol intervention changes observed labels without changing text, (d) the paired effects are stable across seeds and robustness settings, (e) correction/adaptation effects are selective rather than universal, and (f) external stress-test findings are reported with their limitations. If these conditions fail, the paper must be framed as a protocol and negative/diagnostic study rather than as a validated attribution method.

### Claim guardrails

The words **causal**, **platform-only**, **language-only**, **perfect**, and **generalizes to all hate-speech corpora** must not appear in the final paper unless a new design and evidence directly justify them. The primary estimand is an intervention-defined performance decomposition, not an unrestricted causal effect.

In [9]:
# 2. Imports and environment
import os, re, json, math, time, random, hashlib, itertools, warnings, platform, sys, pickle
from pathlib import Path
from dataclasses import dataclass, asdict
from collections import Counter
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score, precision_recall_fscore_support, balanced_accuracy_score,
    roc_auc_score, average_precision_score, confusion_matrix
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
try:
    from IPython.display import display
except Exception:
    def display(x): print(x)

try:
    from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
    TRANSFORMERS_AVAILABLE = True
except Exception as exc:
    TRANSFORMERS_AVAILABLE = False
    print("Transformers is not available yet:", exc)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

Python: 3.12.13
PyTorch: 2.10.0+cu128
Device: cuda


In [10]:
# 2.1 TACL-style plotting defaults (shared by every figure below)
TACL_PALETTE = ["#1B2A4A", "#B5442E", "#3E7C59", "#B08A3E", "#5B5B5B", "#7A5C82"]
# navy / brick-red / muted green / ochre / slate-gray / muted plum -- colorblind-considerate, print-safe

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.color": "#DDDDDD",
    "grid.linewidth": 0.5,
    "axes.axisbelow": True,
    "figure.dpi": 220,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})
sns.set_palette(TACL_PALETTE)

TACL_1COL = (3.3, 2.5)   # inches -- fits a single TACL column
TACL_2COL = (6.75, 2.9)  # inches -- fits the full text width

FIG_DIR = Path(os.environ.get("SHIFTLENS_OUT_DIR", "/kaggle/working/results")) / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("TACL plotting style set. Figures will be written to:", FIG_DIR)


TACL plotting style set. Figures will be written to: /kaggle/working/results/figures


In [11]:
# 3. Research configuration
class CFG:
    # Edit only DATA_ROOT when moving away from Kaggle.
    DATA_ROOT = os.environ.get("SHIFTLENS_DATA_ROOT", "/kaggle/input/datasets/slytherin401533/cyberbullying/data")
    YOUTUBE_CSV = f"{DATA_ROOT}/YouTube-BN/youtube_bn_annotated.csv"
    BDSHS_DIR = f"{DATA_ROOT}/BD-SHS"
    HATEXPLAIN_DIR = f"{DATA_ROOT}/HateXplain"
    BENGALIHATE_DIR = f"{DATA_ROOT}/Bengali Hate"

    # All results (tables, figures, checkpoints, logs, manifests) are written under one named folder.
    OUT_DIR = os.environ.get("SHIFTLENS_OUT_DIR", "/kaggle/working/results")
    BACKBONE = os.environ.get("SHIFTLENS_BACKBONE", "xlm-roberta-base")
    MAX_LENGTH = int(os.environ.get("SHIFTLENS_MAX_LENGTH", "128"))
    TRAIN_FRAC = 0.55
    VAL_FRAC = 0.15
    ADAPT_FRAC = 0.10
    TEST_FRAC = 0.15
    RESERVE_FRAC = 0.05

    # Full paper run. Do not reduce this in the final reported experiment.
    SEEDS = [int(x) for x in os.environ.get("SHIFTLENS_SEEDS", "13,42,2024,3407,9001").split(",") if x.strip()]
    FACTORS = ["content", "protocol"]
    MODELS = [x.strip() for x in os.environ.get("SHIFTLENS_MODELS", "MAJORITY,TFIDF_LOGREG,ERM,DANN,CORAL,PROTOCOL_CORRECTED").split(",") if x.strip()]
    ROBUSTNESS_MODELS = [x.strip() for x in os.environ.get("SHIFTLENS_ROBUSTNESS_MODELS", "TFIDF_LOGREG").split(",") if x.strip()]
    BATCH_SIZE = int(os.environ.get("SHIFTLENS_BATCH_SIZE", "8"))
    EVAL_BATCH_SIZE = int(os.environ.get("SHIFTLENS_EVAL_BATCH_SIZE", "32"))
    GRAD_ACCUM_STEPS = int(os.environ.get("SHIFTLENS_GRAD_ACCUM_STEPS", "4"))
    EPOCHS = int(os.environ.get("SHIFTLENS_EPOCHS", "4"))
    LR = 2e-5
    WEIGHT_DECAY = 0.01
    WARMUP_FRAC = 0.06
    MAX_GRAD_NORM = 1.0
    DOMAIN_WEIGHT = 0.10
    CORAL_WEIGHT = 0.10
    MIXED_PRECISION = os.environ.get("SHIFTLENS_MIXED_PRECISION", "1") == "1"
    NUM_WORKERS = int(os.environ.get("SHIFTLENS_NUM_WORKERS", "2"))

    # Intervention definitions are fixed before looking at test outcomes.
    CONTENT_STRENGTH = 2.0
    CONTENT_TARGET_TOPICS = ["Religious", "Geopolitical", "Gender abusive"]
    # Rows are true class; columns are observed class.
    PROTOCOL_TRANSITION = np.array([[0.90, 0.10], [0.20, 0.80]], dtype=np.float64)

    N_BOOTSTRAP = 5000
    RUN_TRAINING = os.environ.get("SHIFTLENS_RUN_TRAINING", "0") == "1"  # Set True only after the smoke test succeeds.
    RUN_REAL_EVAL = os.environ.get("SHIFTLENS_RUN_REAL_EVAL", "0") == "1"
    RUN_ROBUSTNESS = os.environ.get("SHIFTLENS_RUN_ROBUSTNESS", "0") == "1"
    SMOKE_TEST = os.environ.get("SHIFTLENS_SMOKE_TEST", "1") == "1"

    @staticmethod
    def paths():
        root = Path(CFG.OUT_DIR)
        for sub in ["tables", "figures", "checkpoints", "logs", "manifests"]:
            (root / sub).mkdir(parents=True, exist_ok=True)
        return root

OUT_ROOT = CFG.paths()
CFG.TEST_FRAC = round(1.0 - CFG.TRAIN_FRAC - CFG.VAL_FRAC - CFG.ADAPT_FRAC - CFG.RESERVE_FRAC, 4)
assert abs(CFG.TRAIN_FRAC + CFG.VAL_FRAC + CFG.ADAPT_FRAC + CFG.TEST_FRAC + CFG.RESERVE_FRAC - 1.0) < 1e-9
assert np.allclose(CFG.PROTOCOL_TRANSITION.sum(axis=1), 1.0)
assert len(CFG.SEEDS) >= 5, "The final reported run must use at least five independent seeds."
print(json.dumps({k: v for k, v in vars(CFG).items() if not k.startswith('_') and not callable(v)}, indent=2, default=str))

{
  "DATA_ROOT": "/kaggle/input/datasets/slytherin401533/cyberbullying/data",
  "YOUTUBE_CSV": "/kaggle/input/datasets/slytherin401533/cyberbullying/data/YouTube-BN/youtube_bn_annotated.csv",
  "BDSHS_DIR": "/kaggle/input/datasets/slytherin401533/cyberbullying/data/BD-SHS",
  "HATEXPLAIN_DIR": "/kaggle/input/datasets/slytherin401533/cyberbullying/data/HateXplain",
  "BENGALIHATE_DIR": "/kaggle/input/datasets/slytherin401533/cyberbullying/data/Bengali Hate",
  "OUT_DIR": "/kaggle/working/results",
  "BACKBONE": "xlm-roberta-base",
  "MAX_LENGTH": 128,
  "TRAIN_FRAC": 0.55,
  "VAL_FRAC": 0.15,
  "ADAPT_FRAC": 0.1,
  "TEST_FRAC": 0.15,
  "RESERVE_FRAC": 0.05,
  "SEEDS": [
    13,
    42,
    2024,
    3407,
    9001
  ],
  "FACTORS": [
    "content",
    "protocol"
  ],
  "MODELS": [
    "MAJORITY",
    "TFIDF_LOGREG",
    "ERM",
    "DANN",
    "CORAL",
    "PROTOCOL_CORRECTED"
  ],
  "ROBUSTNESS_MODELS": [
    "TFIDF_LOGREG"
  ],
  "BATCH_SIZE": 8,
  "EVAL_BATCH_SIZE": 32,
  "GRAD_ACCUM

In [12]:
# RESULTS_DIR = "/kaggle/working/results"

In [13]:
# import shutil
# import os

# BACKUP_ROOT = "/kaggle/input/datasets/slytherin401533/backup1/Final Stage_backup"

# shutil.copytree(
#     os.path.join(BACKUP_ROOT, "results"),
#     RESULTS_DIR,
#     dirs_exist_ok=True
# )

# print("Results:", len(os.listdir(RESULTS_DIR)))

In [14]:
# 4. Reproducibility, configuration hashing, and safe artifact writes

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def jsonable_config():
    return {
        "backbone": CFG.BACKBONE,
        "max_length": CFG.MAX_LENGTH,
        "seeds": CFG.SEEDS,
        "factors": CFG.FACTORS,
        "models": CFG.MODELS,
        "train_frac": CFG.TRAIN_FRAC,
        "val_frac": CFG.VAL_FRAC,
        "adapt_frac": CFG.ADAPT_FRAC,
        "test_frac": CFG.TEST_FRAC,
        "content_strength": CFG.CONTENT_STRENGTH,
        "content_target_topics": CFG.CONTENT_TARGET_TOPICS,
        "protocol_transition": CFG.PROTOCOL_TRANSITION.tolist(),
        "epochs": CFG.EPOCHS,
    }

def config_hash():
    payload = json.dumps(jsonable_config(), sort_keys=True).encode()
    return hashlib.sha256(payload).hexdigest()[:16]

def text_hash(text):
    norm = re.sub(r"\s+", " ", str(text).strip().lower())
    return hashlib.sha256(norm.encode("utf-8", "ignore")).hexdigest()[:20]

def atomic_csv(df, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)

def append_deduplicated(df_new, path, keys):
    path = Path(path)
    if path.exists():
        df_old = pd.read_csv(path)
        df = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df = df_new.copy()
    df = df.drop_duplicates(keys, keep="last")
    atomic_csv(df, path)
    return df

RUN_ID = config_hash()
RUN_METADATA = {
    "run_id": RUN_ID,
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.cuda.is_available(),
    "device": str(DEVICE),
    "config": jsonable_config(),
}
(OUT_ROOT / "manifests" / "run_metadata.json").write_text(json.dumps(RUN_METADATA, indent=2, default=str))
set_seed(CFG.SEEDS[0])
print("RUN_ID:", RUN_ID)

RUN_ID: c923a2aa33c46e90


## 5. Data provenance and harmonization

All source data are loaded into one explicit schema. A corpus is not treated as a clean target merely because it has a different name: provenance, label semantics, platform, language/register, and annotation reliability are recorded separately.

The anchor corpus is used to create the synthetic factorial. External corpora are **never** used to choose hyperparameters, thresholds, model checkpoints, intervention strengths, or synthetic topics. They are used only for the held-out real-corpus stress test.

In [15]:
# 5.1 Flexible loaders and label normalization
SCHEMA = ["text", "source", "true_label", "topic", "target_group", "language", "platform", "is_reliable", "official_split"]

POSITIVE = {"1", "true", "yes", "hate", "hateful", "offensive", "abusive", "bullying", "bully", "toxic"}
NEGATIVE = {"0", "false", "no", "normal", "non-abusive", "non abusive", "not bully", "not_bully", "none", "neutral", "clean"}

def parse_binary(values):
    out = []
    for v in values:
        if pd.isna(v):
            out.append(np.nan); continue
        s = str(v).strip().lower()
        try:
            x = float(s)
            out.append(float(x) if x in (0.0, 1.0) else np.nan)
            continue
        except Exception:
            pass
        if s in POSITIVE: out.append(1.0)
        elif s in NEGATIVE: out.append(0.0)
        else: out.append(np.nan)
    return pd.Series(out, index=values.index, dtype="float64")

def pick_col(df, candidates, fallback=None):
    lower = {str(c).lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in lower: return lower[c.lower()]
    return fallback

def standardize(text, source, raw_label=None, topic=None, target_group=None,
                language="bn", platform="unknown", is_reliable=True,
                true_label_override=None, official_split=None):
    n = len(text)
    parsed_labels = (pd.Series(true_label_override).astype(float).reset_index(drop=True)
                     if true_label_override is not None
                     else parse_binary(pd.Series(raw_label if raw_label is not None else [np.nan]*n)).reset_index(drop=True))
    result = pd.DataFrame({
        "text": pd.Series(text).astype(str).str.replace(r"\s+", " ", regex=True).str.strip().reset_index(drop=True),
        "source": source,
        "true_label": parsed_labels,
        "topic": pd.Series(topic if topic is not None else ["__unknown__"]*n).astype(str).reset_index(drop=True),
        "target_group": pd.Series(target_group if target_group is not None else ["__unknown__"]*n).astype(str).reset_index(drop=True),
        "language": pd.Series(language if not isinstance(language, str) else [language]*n).astype(str).reset_index(drop=True),
        "platform": pd.Series(platform if not isinstance(platform, str) else [platform]*n).astype(str).reset_index(drop=True),
        "is_reliable": bool(is_reliable),
        "official_split": pd.Series(official_split if official_split is not None else ["__none__"]*n).astype(str).reset_index(drop=True),
    })
    result["text_hash"] = result["text"].map(text_hash)
    result = result[result["text"].str.len() > 0].reset_index(drop=True)
    if result["is_reliable"].iloc[0] if len(result) else False:
        result = result[result["true_label"].isin([0.0, 1.0])].reset_index(drop=True)
    return result

def read_first_csv(directory, names=None):
    directory = Path(directory)
    if not directory.exists(): raise FileNotFoundError(directory)
    candidates = []
    if names:
        candidates.extend(directory / n for n in names)
    candidates.extend(sorted(directory.glob("*.csv")))
    for p in candidates:
        if p.exists(): return pd.read_csv(p), p
    raise FileNotFoundError(f"No CSV found in {directory}")

def load_youtube():
    df = pd.read_csv(CFG.YOUTUBE_CSV)
    text_col = pick_col(df, ["text", "sentence", "comment"], df.columns[0])
    label_col = pick_col(df, ["label", "category", "class"], df.columns[1])
    raw = df[label_col].astype("string").str.strip()
    # YouTube-BN has one negative category and five annotated abuse/topic categories.
    # Unknown or missing labels are rejected rather than silently converted to positive.
    allowed = {"non-abusive", "religious", "personal", "political", "geopolitical", "gender abusive"}
    observed = set(raw.dropna().str.lower().unique())
    unknown = sorted(observed - allowed)
    if unknown or raw.isna().any() or (raw == "").any():
        raise ValueError(f"Unexpected/missing YouTube labels: unknown={unknown}, missing={int(raw.isna().sum())}")
    binary = raw.str.lower().map({"non-abusive": 0.0}).fillna(1.0).astype(float)
    lang_col = pick_col(df, ["lang", "language"])
    return standardize(df[text_col], "YouTube", raw_label=raw, topic=raw,
                       language=df[lang_col] if lang_col else "bn",
                       platform="YouTube", is_reliable=True,
                       true_label_override=binary)

def load_bdshs():
    parts = []
    for name in ["train.csv", "val.csv", "validation.csv", "test.csv"]:
        p = Path(CFG.BDSHS_DIR) / name
        if p.exists():
            part = pd.read_csv(p)
            part["__official_split"] = Path(name).stem
            parts.append(part)
    if not parts: raise FileNotFoundError(CFG.BDSHS_DIR)
    df = pd.concat(parts, ignore_index=True)
    text_col = pick_col(df, ["sentence", "text", "comment"], df.columns[0])
    label_col = pick_col(df, ["hate speech", "hate_speech", "label", "class"], df.columns[1])
    topic_col = pick_col(df, ["type", "category", "topic"])
    target_col = pick_col(df, ["target", "target_group"])
    return standardize(df[text_col], "BD-SHS", raw_label=df[label_col],
                       topic=df[topic_col] if topic_col else None,
                       target_group=df[target_col] if target_col else None,
                       language="bn", platform="Facebook", is_reliable=True,
                       official_split=df["__official_split"])

def load_hatexplain():
    parts = []
    for name in ["train.csv", "validation.csv", "val.csv", "test.csv"]:
        p = Path(CFG.HATEXPLAIN_DIR) / name
        if p.exists():
            part = pd.read_csv(p)
            part["__official_split"] = Path(name).stem
            parts.append(part)
    if not parts: raise FileNotFoundError(CFG.HATEXPLAIN_DIR)
    df = pd.concat(parts, ignore_index=True)
    text_col = pick_col(df, ["text", "post", "comment"], df.columns[0])
    label_col = pick_col(df, ["label", "class", "gold_label"], df.columns[1])
    raw = df[label_col].astype(str).str.strip().str.lower()
    # Offensive is intentionally excluded from the binary external metric;
    # normal=0 and hatespeech=1 retain the cleanest semantic correspondence.
    binary = raw.map({"normal": 0.0, "hatespeech": 1.0})
    return standardize(df[text_col], "HateXplain", raw_label=raw,
                       true_label_override=binary,
                       language="en/code-mixed", platform="Twitter/Gab", is_reliable=True,
                       official_split=df["__official_split"])

def load_bengalihate(filename, source):
    p = Path(CFG.BENGALIHATE_DIR) / filename
    if not p.exists(): raise FileNotFoundError(p)
    df = pd.read_csv(p, sep=None, engine="python")
    text_col = pick_col(df, ["text", "sentence", "comment"], df.columns[0])
    label_col = pick_col(df, ["label", "class", "category"], df.columns[1] if len(df.columns) > 1 else None)
    raw = df[label_col] if label_col else pd.Series([np.nan]*len(df))
    parsed = parse_binary(raw)
    reliable = parsed.notna().all() and parsed.nunique() == 2
    # BengaliHate resources are retained as external targets; if labels are
    # pure-positive/topic-only they are not used for classification metrics.
    return standardize(df[text_col], source, raw_label=raw if reliable else None,
                       topic=df[label_col] if label_col else None,
                       language="bn", platform="mixed/unknown", is_reliable=reliable,
                       true_label_override=parsed if reliable else None)

def load_all():
    loaders = {
        "YouTube": load_youtube,
        "BD-SHS": load_bdshs,
        "HateXplain": load_hatexplain,
        "BengaliHate_v1": lambda: load_bengalihate("bengali_hate_v1.0.csv", "BengaliHate_v1"),
        "BengaliHate_v2": lambda: load_bengalihate("bengali_hate_v2.0.csv", "BengaliHate_v2"),
    }
    raw, errors = {}, {}
    for name, fn in loaders.items():
        try: raw[name] = fn()
        except Exception as exc: errors[name] = str(exc)
    return raw, errors

RAW, LOAD_ERRORS = load_all()
if LOAD_ERRORS: print("Load notes:", json.dumps(LOAD_ERRORS, indent=2))
if RAW:
    FULL = pd.concat(RAW.values(), ignore_index=True)
    print(FULL.groupby(["source", "is_reliable"], dropna=False).size().rename("n"))
else:
    FULL = pd.DataFrame(columns=SCHEMA + ["text_hash"])

source          is_reliable
BD-SHS          True           50281
BengaliHate_v1  False           3418
BengaliHate_v2  False           5698
HateXplain      True           13749
YouTube         True           67982
Name: n, dtype: int64


In [16]:
# 5.2 Validation, provenance, and cross-corpus overlap audit

def validate_corpus(df):
    assert df["text"].notna().all(), "Missing text detected."
    assert (df["text"].str.len() > 0).all(), "Empty text detected."
    assert df["text_hash"].notna().all()
    reliable = df[df["is_reliable"]]
    if len(reliable):
        assert reliable["true_label"].isin([0.0, 1.0]).all(), "Reliable labels must be binary."
        assert reliable["true_label"].nunique() == 2, "Reliable corpus must contain both classes."
    return {
        "n": int(len(df)),
        "n_unique_text": int(df["text_hash"].nunique()),
        "duplicate_rate": float(1 - df["text_hash"].nunique() / max(len(df), 1)),
        "reliable": bool(df["is_reliable"].all()),
        "label_coverage": float(df["true_label"].notna().mean()),
        "positive_rate": float(df["true_label"].mean()) if df["true_label"].notna().any() else np.nan,
    }

SUMMARY = []
for source, df in RAW.items():
    try: report = validate_corpus(df)
    except AssertionError as exc: report = {"n": len(df), "validation_error": str(exc)}
    report["source"] = source
    SUMMARY.append(report)
summary_df = pd.DataFrame(SUMMARY)
display(summary_df)
summary_df.to_csv(OUT_ROOT / "tables" / "corpus_summary.csv", index=False)

# Remove exact text overlap with YouTube from external target pools.
anchor_hashes = set(RAW.get("YouTube", pd.DataFrame(columns=["text_hash"]))["text_hash"])
OVERLAP = []
for source, df in RAW.items():
    if source == "YouTube": continue
    n_overlap = int(df["text_hash"].isin(anchor_hashes).sum())
    OVERLAP.append({"source": source, "overlap_with_anchor": n_overlap, "rate": n_overlap / max(len(df), 1)})
OVERLAP_DF = pd.DataFrame(OVERLAP)
display(OVERLAP_DF)
OVERLAP_DF.to_csv(OUT_ROOT / "tables" / "anchor_overlap.csv", index=False)

,n,n_unique_text,duplicate_rate,reliable,label_coverage,positive_rate,source
0,67982,67981,0.000015,True,1.0,0.515489,YouTube
1,50281,50276,0.000099,True,1.0,0.480420,BD-SHS
2,13749,13724,0.001818,True,1.0,0.431668,HateXplain
3,3418,3176,0.070802,False,0.0,NaN,BengaliHate_v1
4,5698,5694,0.000702,False,0.0,NaN,BengaliHate_v2


,source,overlap_with_anchor,rate
0,BD-SHS,169,0.003361
1,HateXplain,0,0.000000
2,BengaliHate_v1,1,0.000293
3,BengaliHate_v2,0,0.000000


## 6. Grouped anchor split and leakage prevention

The anchor corpus is split by normalized text hash, not by raw row. This prevents duplicated comments or near-identical rows from crossing train, adaptation, validation, and evaluation sets. The split is performed once with a declared seed and saved as a manifest. The evaluation labels are never used for model selection.

In [17]:
# 6.1 Group-stratified split for the labeled anchor corpus

def group_stratified_split(df, fractions, seed):
    df = df.copy().reset_index(drop=True)
    assert len(fractions) == 5 and abs(sum(fractions) - 1.0) < 1e-8
    label_nunique = df.groupby("text_hash")["true_label"].nunique(dropna=False)
    assert (label_nunique <= 1).all(), "Conflicting labels within an exact-text group require manual adjudication."
    group = df.groupby("text_hash", as_index=False)["true_label"].first()
    remaining_groups = group["text_hash"].to_numpy()
    remaining_y = group["true_label"].to_numpy().astype(int)
    names = ["train", "val", "adapt", "test", "reserve"]
    split_groups = {}
    for i, name in enumerate(names[:-1]):
        share = fractions[i] / sum(fractions[i:])
        chosen, remaining_groups, _, remaining_y = train_test_split(
            remaining_groups, remaining_y, train_size=share,
            random_state=seed + i, stratify=remaining_y
        )
        split_groups[name] = set(chosen)
    split_groups["reserve"] = set(remaining_groups)
    result = {name: df[df["text_hash"].isin(hashes)].copy().reset_index(drop=True)
              for name, hashes in split_groups.items()}
    assert set.union(*split_groups.values()) == set(group["text_hash"])
    assert sum(len(x) for x in split_groups.values()) == len(split_groups["train"] | split_groups["val"] | split_groups["adapt"] | split_groups["test"] | split_groups["reserve"])
    assert all(len(split_groups[a] & split_groups[b]) == 0 for i, a in enumerate(names) for b in names[i+1:])
    return result

require_anchor = "YouTube" in RAW and RAW["YouTube"]["true_label"].notna().all()
if not require_anchor:
    raise RuntimeError("The YouTube anchor corpus with reliable binary labels is required. Check CFG.DATA_ROOT and loader mappings.")

ANCHOR = RAW["YouTube"].copy()
ANCHOR_SPLITS = group_stratified_split(ANCHOR, [CFG.TRAIN_FRAC, CFG.VAL_FRAC, CFG.ADAPT_FRAC, CFG.TEST_FRAC, CFG.RESERVE_FRAC], CFG.SEEDS[0])
SPLIT_MANIFEST = []
for split_name, split_df in ANCHOR_SPLITS.items():
    split_df = split_df.copy()
    split_df["split"] = split_name
    SPLIT_MANIFEST.append(split_df[["text_hash", "source", "true_label", "split"]])
SPLIT_MANIFEST_DF = pd.concat(SPLIT_MANIFEST, ignore_index=True)
SPLIT_MANIFEST_DF.to_csv(OUT_ROOT / "manifests" / "anchor_split_manifest.csv", index=False)
print({k: len(v) for k, v in ANCHOR_SPLITS.items()})
print(pd.DataFrame({k: v["true_label"].mean() for k, v in ANCHOR_SPLITS.items()}, index=["positive_rate"]))

{'train': 37390, 'val': 10197, 'adapt': 6798, 'test': 10197, 'reserve': 3400}
                  train       val     adapt      test   reserve
positive_rate  0.515485  0.515446  0.515446  0.515544  0.515588


In [18]:
# 6.2 External target splits: labels are used only for final evaluation when available.
def external_target_split(df, seed):
    df = df.copy().reset_index(drop=True)
    if "official_split" in df.columns and (df["official_split"].astype(str).str.lower() == "test").any():
        test = df[df["official_split"].astype(str).str.lower() == "test"].copy()
        adapt = df[df["official_split"].astype(str).str.lower() != "test"].copy()
        overlap = set(adapt["text_hash"]) & set(test["text_hash"])
        if overlap:
            # Preserve the official test set and remove duplicate texts from adaptation.
            adapt = adapt[~adapt["text_hash"].isin(overlap)].copy()
            print(f"External official split overlap: dropped {len(overlap)} text groups from adapt; official test preserved.")
        return {"adapt": adapt.reset_index(drop=True), "test": test.reset_index(drop=True)}
    if len(df) < 10:
        return {"adapt": df.iloc[0:0].copy(), "test": df.copy()}
    group_df = df[["text_hash", "true_label"]].drop_duplicates("text_hash").reset_index(drop=True)
    if group_df["true_label"].nunique() < 2:
        adapt_hash, test_hash = train_test_split(group_df["text_hash"].to_numpy(), test_size=0.50, random_state=seed)
    else:
        adapt_hash, test_hash = train_test_split(
            group_df["text_hash"].to_numpy(),
            test_size=0.50,
            random_state=seed,
            stratify=group_df["true_label"].to_numpy(),
        )
    adapt_set, test_set = set(adapt_hash), set(test_hash)
    assert not adapt_set & test_set
    return {
        "adapt": df[df["text_hash"].isin(adapt_set)].copy().reset_index(drop=True),
        "test": df[df["text_hash"].isin(test_set)].copy().reset_index(drop=True),
    }

REAL_TARGET_SPLITS = {}
for source, df in RAW.items():
    if source == "YouTube":
        continue
    clean = df[~df["text_hash"].isin(anchor_hashes)].copy()
    REAL_TARGET_SPLITS[source] = external_target_split(clean, CFG.SEEDS[0])
print({k: {s: len(v) for s, v in d.items()} for k, d in REAL_TARGET_SPLITS.items()})


External official split overlap: dropped 2 text groups from adapt; official test preserved.
External official split overlap: dropped 2 text groups from adapt; official test preserved.
{'BD-SHS': {'adapt': 45097, 'test': 5013}, 'HateXplain': {'adapt': 12369, 'test': 1376}, 'BengaliHate_v1': {'adapt': 1725, 'test': 1692}, 'BengaliHate_v2': {'adapt': 2849, 'test': 2849}}


## 7. Identifiable intervention design

The factorial has exactly two factors because only two interventions can be isolated without inventing a false platform or language transformation.

### Content intervention

For each class separately, the target evaluation/adaptation pools are re-sampled without replacement from a disjoint adaptation/test/reserve candidate pool. The intervention changes only topic-composition weights, using topics already annotated in the anchor corpus; class prior, pool sizes, and hash-level disjointness are preserved. The operator is deterministic given the seed and does not create synthetic text.

### Protocol intervention

The source training labels are corrupted using a declared class-conditional transition matrix. The latent clean labels remain available only for audit and clean evaluation. The model sees `observed_label`, not `true_label`. This creates a known annotation-protocol shift without changing the text distribution.

### External shift

BD-SHS, HateXplain, and BengaliHate are not assigned synthetic factor labels. They are external stress tests. The notebook reports descriptive shift signatures and out-of-distribution flags, but it does not compute a causal or Shapley attribution for them.

In [19]:
# 7.1 Intervention operators

def content_shift_pools(base_adapt, base_test, reserve, on, seed, strength=None):
    if not on:
        return base_adapt.copy().reset_index(drop=True), base_test.copy().reset_index(drop=True)
    strength = CFG.CONTENT_STRENGTH if strength is None else float(strength)
    candidate = pd.concat([base_adapt, base_test, reserve], ignore_index=True)
    rng = np.random.default_rng(seed)
    adapt_parts, test_parts = [], []
    for y in [0, 1]:
        sub = candidate[candidate["true_label"].astype(int) == y].copy()
        n_adapt = int((base_adapt["true_label"].astype(int) == y).sum())
        n_test = int((base_test["true_label"].astype(int) == y).sum())
        assert len(sub) >= n_adapt + n_test
        topic = sub["topic"].fillna("__unknown__").astype(str)
        weights = np.ones(len(sub), dtype=float)
        weights[topic.isin(CFG.CONTENT_TARGET_TOPICS).to_numpy()] *= (1.0 + strength)
        weights = weights / weights.sum()
        chosen = rng.choice(np.arange(len(sub)), size=n_adapt + n_test, replace=False, p=weights)
        picked = sub.iloc[chosen].reset_index(drop=True)
        adapt_parts.append(picked.iloc[:n_adapt])
        test_parts.append(picked.iloc[n_adapt:n_adapt+n_test])
    adapt = pd.concat(adapt_parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    test = pd.concat(test_parts, ignore_index=True).sample(frac=1.0, random_state=seed + 1).reset_index(drop=True)
    assert len(adapt) == len(base_adapt) and len(test) == len(base_test)
    assert adapt["true_label"].value_counts().sort_index().equals(base_adapt["true_label"].value_counts().sort_index())
    assert test["true_label"].value_counts().sort_index().equals(base_test["true_label"].value_counts().sort_index())
    assert set(adapt["text_hash"]).isdisjoint(set(test["text_hash"]))
    return adapt, test

def protocol_shift(df, on, seed):
    out = df.copy().reset_index(drop=True)
    out["observed_label"] = out["true_label"].astype(float)
    if not on: return out
    rng = np.random.default_rng(seed)
    y = out["true_label"].astype(int).to_numpy()
    observed = [rng.choice([0, 1], p=CFG.PROTOCOL_TRANSITION[yi]) for yi in y]
    out["observed_label"] = np.asarray(observed, dtype=float)
    out["protocol_on"] = 1
    return out

def build_factorial_cell(content_on, protocol_on, seed):
    train = protocol_shift(ANCHOR_SPLITS["train"], protocol_on, seed + 101)
    val = protocol_shift(ANCHOR_SPLITS["val"], protocol_on, seed + 102)
    adapt, test = content_shift_pools(ANCHOR_SPLITS["adapt"], ANCHOR_SPLITS["test"], ANCHOR_SPLITS["reserve"], content_on, seed + 103)
    for frame in [train, val]: frame["observed_label"] = frame["observed_label"].astype(int)
    return {
        "train": train, "val": val, "adapt": adapt,
        "test": test, "content_on": int(content_on), "protocol_on": int(protocol_on), "seed": int(seed)
    }

# Intervention unit tests before any model is trained.
for seed in CFG.SEEDS[:2]:
    for c, p in itertools.product([0, 1], repeat=2):
        cell = build_factorial_cell(c, p, seed)
        assert len(cell["train"]) == len(ANCHOR_SPLITS["train"])
        assert len(cell["test"]) == len(ANCHOR_SPLITS["test"])
        assert cell["test"]["true_label"].mean() == ANCHOR_SPLITS["test"]["true_label"].mean()
        assert set(cell["adapt"]["text_hash"]).isdisjoint(set(cell["test"]["text_hash"]))
        assert set(cell["adapt"]["text_hash"]).isdisjoint(set(cell["train"]["text_hash"]))
        assert set(cell["test"]["text_hash"]).isdisjoint(set(cell["train"]["text_hash"]))
        if p == 0:
            assert np.array_equal(cell["train"]["observed_label"], cell["train"]["true_label"])
print("Intervention unit tests passed.")

Intervention unit tests passed.


In [20]:
# 7.2 Intervention audit table
AUDIT = []
for c, p in itertools.product([0, 1], repeat=2):
    cell = build_factorial_cell(c, p, CFG.SEEDS[0])
    AUDIT.append({
        "content_on": c, "protocol_on": p,
        "train_n": len(cell["train"]), "test_n": len(cell["test"]),
        "train_true_pos_rate": cell["train"]["true_label"].mean(),
        "train_observed_pos_rate": cell["train"]["observed_label"].mean(),
        "test_pos_rate": cell["test"]["true_label"].mean(),
        "test_topic_entropy": float(-(cell["test"]["topic"].value_counts(normalize=True) * np.log(cell["test"]["topic"].value_counts(normalize=True) + 1e-12)).sum()),
    })
AUDIT_DF = pd.DataFrame(AUDIT)
display(AUDIT_DF)
AUDIT_DF.to_csv(OUT_ROOT / "tables" / "intervention_audit.csv", index=False)

,content_on,protocol_on,train_n,test_n,train_true_pos_rate,train_observed_pos_rate,test_pos_rate,test_topic_entropy
0,0,0,37390,10197,0.515485,0.515485,0.515544,1.502117
1,0,1,37390,10197,0.515485,0.459160,0.515544,1.502117
2,1,0,37390,10197,0.515485,0.515485,0.515544,1.503953
3,1,1,37390,10197,0.515485,0.459160,0.515544,1.503953


## 8. Models and losses

The original custom correction and contrastive objectives are replaced with transparent baselines:

- **MAJORITY:** deterministic label-prior floor.
- **TFIDF_LOGREG:** a lexical non-neural reference fit on source training text only.
- **ERM:** supervised empirical-risk minimization on observed source labels.
- **DANN:** standard domain-adversarial training using source labels and target text without target labels.
- **CORAL:** covariance alignment between source and target representations.
- **PROTOCOL_CORRECTED:** forward loss correction using the pre-specified true-to-observed transition matrix. This is only a synthetic protocol arm; it is not applied to real targets with unknown transition matrices.

No model is selected using target labels. Checkpoints are selected only by source validation macro-F1, with a deterministic tie-break on source validation loss.

In [21]:
# 8.1 Tokenization and datasets
NEURAL_MODELS = {"ERM", "DANN", "CORAL", "PROTOCOL_CORRECTED"}
NEURAL_REQUESTED = any(m in NEURAL_MODELS for m in CFG.MODELS)
if NEURAL_REQUESTED and not TRANSFORMERS_AVAILABLE:
    raise RuntimeError("Neural models are configured but transformers is unavailable. Install transformers and restart the runtime.")
TOKENIZER = AutoTokenizer.from_pretrained(CFG.BACKBONE, use_fast=True) if NEURAL_REQUESTED else None
if NEURAL_REQUESTED and TOKENIZER is None:
    raise RuntimeError("Neural models are configured but tokenizer initialization returned None.")

class TextDataset(Dataset):
    def __init__(self, df, label_col="observed_label", domain=0):
        self.text = df["text"].astype(str).tolist()
        self.labels = df[label_col].fillna(-1).astype(int).tolist() if label_col in df else [-1] * len(df)
        self.domain = [int(domain)] * len(df)
    def __len__(self): return len(self.text)
    def __getitem__(self, i): return self.text[i], self.labels[i], self.domain[i]

def make_collate(tokenizer):
    if tokenizer is None:
        raise RuntimeError("A tokenizer is required for neural DataLoader construction.")
    def collate(batch):
        texts, labels, domains = zip(*batch)
        tok = tokenizer(list(texts), padding=True, truncation=True, max_length=CFG.MAX_LENGTH, return_tensors="pt")
        return {"input_ids": tok["input_ids"], "attention_mask": tok["attention_mask"],
                "labels": torch.tensor(labels, dtype=torch.long), "domains": torch.tensor(domains, dtype=torch.long)}
    return collate

COLLATE = make_collate(TOKENIZER) if NEURAL_REQUESTED else None

def make_loader(df, label_col="observed_label", domain=0, shuffle=False, batch_size=None):
    if COLLATE is None:
        raise RuntimeError("Neural DataLoader requested while tokenizer is disabled. Check CFG.MODELS and execute cells from the beginning.")
    ds = TextDataset(df, label_col=label_col, domain=domain)
    return DataLoader(ds, batch_size=batch_size or CFG.BATCH_SIZE, shuffle=shuffle,
                      num_workers=CFG.NUM_WORKERS, pin_memory=torch.cuda.is_available(), collate_fn=COLLATE)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [22]:
# 8.2 Encoder, gradient reversal, and adaptation losses
class GradientReversal(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None

def grl(x, lambd): return GradientReversal.apply(x, lambd)

class ShiftModel(nn.Module):
    def __init__(self, backbone_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(backbone_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, 2)
        self.domain_classifier = nn.Sequential(nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Linear(hidden // 2, 2))
    def forward(self, input_ids, attention_mask, adv_lambda=0.0):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp_min(1.0)
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        domain_logits = self.domain_classifier(grl(pooled, adv_lambda))
        return logits, pooled, domain_logits

def coral_loss(source, target):
    if source.size(0) < 2 or target.size(0) < 2: return source.new_tensor(0.0)
    def cov(x):
        x = x - x.mean(0, keepdim=True)
        return (x.T @ x) / max(x.size(0) - 1, 1)
    cs, ct = cov(source), cov(target)
    return ((cs - ct) ** 2).mean() / (4 * source.size(1) ** 2)

def forward_corrected_loss(logits, observed_labels, transition):
    # transition[row=true, col=observed]. P(observed | x) = P(true | x) @ T.
    probs_true = logits.softmax(-1).clamp_min(1e-7)
    probs_observed = probs_true @ transition
    return F.nll_loss(torch.log(probs_observed.clamp_min(1e-7)), observed_labels)

In [23]:
# 8.3 Metrics and evaluation

def expected_calibration_error(y_true, prob_pos, n_bins=10):
    y_true, prob_pos = np.asarray(y_true), np.asarray(prob_pos)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (prob_pos >= lo) & (prob_pos < hi if hi < 1 else prob_pos <= hi)
        if mask.any():
            ece += mask.mean() * abs(y_true[mask].mean() - prob_pos[mask].mean())
    return float(ece)

def select_threshold(y_true, prob_pos):
    y_true = np.asarray(y_true).astype(int)
    prob_pos = np.asarray(prob_pos, dtype=float)
    keep = np.isfinite(prob_pos) & np.isfinite(y_true)
    y_true, prob_pos = y_true[keep], prob_pos[keep]
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        return 0.5
    candidates = np.unique(np.r_[np.linspace(0.05, 0.95, 19), prob_pos])
    scores = [f1_score(y_true, (prob_pos >= t).astype(int), average="macro", zero_division=0) for t in candidates]
    return float(candidates[int(np.argmax(scores))])

def metric_dict(y_true, prob_pos, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    prob_pos = np.asarray(prob_pos, dtype=float)
    pred = (prob_pos >= float(threshold)).astype(int)
    out = {
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "f1_pos": f1_score(y_true, pred, pos_label=1, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "ece": expected_calibration_error(y_true, prob_pos),
    }
    out["auroc"] = roc_auc_score(y_true, prob_pos) if len(np.unique(y_true)) == 2 else np.nan
    out["auprc"] = average_precision_score(y_true, prob_pos) if y_true.sum() > 0 else np.nan
    return {k: float(v) for k, v in out.items()}

@torch.no_grad()
def evaluate_model(model, loader, threshold=0.5):
    model.eval(); ys, ps = [], []
    for batch in loader:
        ids = batch["input_ids"].to(DEVICE); mask = batch["attention_mask"].to(DEVICE)
        logits, _, _ = model(ids, mask, adv_lambda=0.0)
        prob = logits.softmax(-1)[:, 1].detach().cpu().numpy()
        labels = batch["labels"].cpu().numpy()
        keep = labels >= 0
        ys.extend(labels[keep]); ps.extend(prob[keep])
    if not ys: return {}, np.array([]), np.array([])
    return metric_dict(np.array(ys), np.array(ps), threshold=threshold), np.array(ys), np.array(ps)

@torch.no_grad()
def evaluate_observed_loss(model, loader):
    model.eval(); losses = []
    for batch in loader:
        ids = batch["input_ids"].to(DEVICE); mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        logits, _, _ = model(ids, mask, adv_lambda=0.0)
        keep = labels >= 0
        if keep.any(): losses.append(float(F.cross_entropy(logits[keep], labels[keep]).detach().cpu()))
    return float(np.mean(losses)) if losses else np.inf


In [24]:
# 8.4 Training loop with source-only checkpoint selection and resumable artifacts

def infinite_loader(loader):
    while True:
        for batch in loader: yield batch

def move_batch(batch):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}

def train_classical_run(model_kind, cell, seed, run_name=None, persist=True, result_path=None):
    set_seed(seed)
    run_name = run_name or f"{model_kind}_c{cell['content_on']}_p{cell['protocol_on']}_s{seed}"
    result_path = Path(result_path or (OUT_ROOT / "tables" / "factorial_results.csv"))
    train_df, val_df, test_df = cell["train"], cell["val"], cell["test"]
    y_train = train_df["observed_label"].astype(int).to_numpy()
    y_test = test_df["true_label"].astype(int).to_numpy()
    if model_kind == "MAJORITY":
        val_prob = np.full(len(val_df), float(y_train.mean()))
        prob = np.full(len(test_df), float(y_train.mean()))
        threshold = select_threshold(val_df["true_label"].astype(int).to_numpy(), val_prob)
        model_bundle = {"kind": model_kind, "positive_rate": float(y_train.mean()), "threshold": float(threshold)}
    else:
        vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=100000, sublinear_tf=True)
        X_train = vectorizer.fit_transform(train_df["text"].astype(str))
        X_val = vectorizer.transform(val_df["text"].astype(str))
        if np.unique(y_train).size < 2:
            raise RuntimeError("TFIDF_LOGREG requires both observed classes in the training split.")
        estimator = LogisticRegression(max_iter=500, solver="liblinear", random_state=seed)
        estimator.fit(X_train, y_train)
        val_prob = estimator.predict_proba(X_val)[:, 1]
        threshold = select_threshold(val_df["true_label"].astype(int).to_numpy(), val_prob)
        prob = estimator.predict_proba(vectorizer.transform(test_df["text"].astype(str)))[:, 1]
        model_bundle = {"kind": model_kind, "vectorizer": vectorizer, "estimator": estimator, "threshold": float(threshold)}
    result = {
        "run_id": RUN_ID, "run_name": run_name, "model": model_kind, "seed": seed,
        "content_on": cell["content_on"], "protocol_on": cell["protocol_on"],
        "best_epoch": 0, "n_train": len(train_df), "n_test": len(test_df),
        "decision_threshold": float(threshold),
        **metric_dict(y_test, prob, threshold=threshold),
    }
    with open(OUT_ROOT / "checkpoints" / f"{run_name}.pkl", "wb") as f: pickle.dump({"model": model_bundle, "result": result}, f)
    if persist:
        append_deduplicated(pd.DataFrame([result]), result_path, ["run_id", "model", "seed", "content_on", "protocol_on"])
    return result

def train_one_run(model_kind, cell, seed, run_name=None, persist=True, result_path=None):
    if model_kind in {"MAJORITY", "TFIDF_LOGREG"}:
        return train_classical_run(model_kind, cell, seed, run_name=run_name, persist=persist, result_path=result_path)
    set_seed(seed)
    run_name = run_name or f"{model_kind}_c{cell['content_on']}_p{cell['protocol_on']}_s{seed}"
    ckpt_path = OUT_ROOT / "checkpoints" / f"{run_name}.pt"
    result_path = Path(result_path or (OUT_ROOT / "tables" / "factorial_results.csv"))

    transition = torch.tensor(CFG.PROTOCOL_TRANSITION if cell["protocol_on"] else np.eye(2), dtype=torch.float32, device=DEVICE)
    train_loader = make_loader(cell["train"], "observed_label", domain=0, shuffle=True)
    # Checkpoint selection uses only the labels observable under the cell's protocol.
    # A clean-label validation score is reported separately but never used for selection.
    val_observed_loader = make_loader(cell["val"], "observed_label", domain=0, shuffle=False, batch_size=CFG.EVAL_BATCH_SIZE)
    val_clean_loader = make_loader(cell["val"].assign(observed_label=cell["val"]["true_label"]), "observed_label", domain=0, shuffle=False, batch_size=CFG.EVAL_BATCH_SIZE)
    adapt_loader = make_loader(cell["adapt"].assign(observed_label=-1), "observed_label", domain=1, shuffle=True)
    test_loader = make_loader(cell["test"].assign(observed_label=cell["test"]["true_label"]), "observed_label", domain=1, shuffle=False, batch_size=CFG.EVAL_BATCH_SIZE)

    model = ShiftModel(CFG.BACKBONE).to(DEVICE)
    optimiser = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    total_steps = max(1, CFG.EPOCHS * math.ceil(len(train_loader) / CFG.GRAD_ACCUM_STEPS))
    scheduler = get_linear_schedule_with_warmup(optimiser, int(CFG.WARMUP_FRAC * total_steps), total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=CFG.MIXED_PRECISION and torch.cuda.is_available())
    target_iter = infinite_loader(adapt_loader)
    best = {"macro_f1": -np.inf, "val_loss": np.inf, "state_dict": None, "epoch": -1, "threshold": 0.5}
    history = []

    for epoch in range(CFG.EPOCHS):
        model.train(); optimiser.zero_grad(set_to_none=True); running = []
        for step, source_batch in enumerate(train_loader):
            source_batch = move_batch(source_batch)
            target_batch = move_batch(next(target_iter))
            with torch.cuda.amp.autocast(enabled=CFG.MIXED_PRECISION and torch.cuda.is_available()):
                slogits, sh, sdomain = model(source_batch["input_ids"], source_batch["attention_mask"], adv_lambda=1.0 if model_kind == "DANN" else 0.0)
                if model_kind == "PROTOCOL_CORRECTED":
                    label_loss = forward_corrected_loss(slogits, source_batch["labels"], transition)
                else:
                    label_loss = F.cross_entropy(slogits, source_batch["labels"])
                loss = label_loss
                if model_kind == "DANN":
                    tlogits, th, tdomain = model(target_batch["input_ids"], target_batch["attention_mask"], adv_lambda=1.0)
                    domains = torch.cat([source_batch["domains"], target_batch["domains"]])
                    dloss = F.cross_entropy(torch.cat([sdomain, tdomain]), domains)
                    loss = loss + CFG.DOMAIN_WEIGHT * dloss
                elif model_kind == "CORAL":
                    _, th, _ = model(target_batch["input_ids"], target_batch["attention_mask"], adv_lambda=0.0)
                    loss = loss + CFG.CORAL_WEIGHT * coral_loss(sh, th)
                loss = loss / CFG.GRAD_ACCUM_STEPS
            scaler.scale(loss).backward()
            if (step + 1) % CFG.GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimiser)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.MAX_GRAD_NORM)
                scaler.step(optimiser); scaler.update(); scheduler.step(); optimiser.zero_grad(set_to_none=True)
            running.append(float(loss.detach().cpu()) * CFG.GRAD_ACCUM_STEPS)

        _, val_observed_y, val_observed_prob = evaluate_model(model, val_observed_loader, threshold=0.5)
        _, val_clean_y, val_clean_prob = evaluate_model(model, val_clean_loader, threshold=0.5)
        threshold = select_threshold(val_observed_y, val_observed_prob)
        val_observed_metrics = metric_dict(val_observed_y, val_observed_prob, threshold=threshold)
        val_clean_metrics = metric_dict(val_clean_y, val_clean_prob, threshold=threshold)
        val_loss = evaluate_observed_loss(model, val_observed_loader)
        row = {"epoch": epoch, "train_loss": np.mean(running),
               **{f"val_observed_{k}": v for k, v in val_observed_metrics.items()},
               "decision_threshold": threshold,
               **{f"val_clean_{k}": v for k, v in val_clean_metrics.items()}}
        history.append(row)
        score = val_observed_metrics.get("macro_f1", -np.inf)
        if score > best["macro_f1"] or (score == best["macro_f1"] and val_loss < best["val_loss"]):
            best = {"macro_f1": score, "val_loss": val_loss, "state_dict": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, "epoch": epoch, "threshold": float(threshold)}

    if best["state_dict"] is None: raise RuntimeError(f"No checkpoint selected for {run_name}")
    model.load_state_dict(best["state_dict"])
    test_metrics, y_true, prob = evaluate_model(model, test_loader, threshold=best["threshold"])
    result = {
        "run_id": RUN_ID, "run_name": run_name, "model": model_kind, "seed": seed,
        "content_on": cell["content_on"], "protocol_on": cell["protocol_on"],
        "best_epoch": best["epoch"], "n_train": len(cell["train"]), "n_test": len(cell["test"]),
        "decision_threshold": float(best["threshold"]),
        **test_metrics,
    }
    torch.save({"state_dict": best["state_dict"], "history": history, "result": result}, ckpt_path)
    if persist:
        append_deduplicated(pd.DataFrame([result]), result_path, ["run_id", "model", "seed", "content_on", "protocol_on"])
    return result

In [25]:
# 8.5 Shared prediction-reload utility with threshold provenance

def reload_predictions(model_kind, seed, content_on, protocol_on):
    run_name = f"{model_kind}_c{content_on}_p{protocol_on}_s{seed}"
    pkl_path = OUT_ROOT / "checkpoints" / f"{run_name}.pkl"
    pt_path = OUT_ROOT / "checkpoints" / f"{run_name}.pt"
    cell = build_factorial_cell(content_on, protocol_on, seed)
    test_df = cell["test"].reset_index(drop=True)

    if model_kind in {"MAJORITY", "TFIDF_LOGREG"}:
        if not pkl_path.exists():
            return None
        with open(pkl_path, "rb") as f:
            bundle = pickle.load(f)
        threshold = float(bundle["model"].get("threshold", bundle.get("result", {}).get("decision_threshold", 0.5)))
        if model_kind == "MAJORITY":
            prob = np.full(len(test_df), bundle["model"]["positive_rate"])
        else:
            vec, est = bundle["model"]["vectorizer"], bundle["model"]["estimator"]
            prob = est.predict_proba(vec.transform(test_df["text"].astype(str)))[:, 1]
    else:
        if not pt_path.exists() or TOKENIZER is None:
            return None
        bundle = torch.load(pt_path, map_location=DEVICE, weights_only=False)
        model = ShiftModel(CFG.BACKBONE).to(DEVICE)
        model.load_state_dict(bundle["state_dict"])
        loader = make_loader(test_df.assign(observed_label=test_df["true_label"]), "observed_label", domain=1, shuffle=False, batch_size=CFG.EVAL_BATCH_SIZE)
        _, y_true, prob = evaluate_model(model, loader, threshold=0.5)
        threshold = float(bundle["result"].get("decision_threshold", 0.5))

    out = test_df.copy()
    out["y_true"] = test_df["true_label"].astype(int)
    out["prob_pos"] = prob
    out["decision_threshold"] = threshold
    out["y_pred"] = (out["prob_pos"] >= threshold).astype(int)
    out["error_type"] = np.select(
        [(out.y_true == 0) & (out.y_pred == 1), (out.y_true == 1) & (out.y_pred == 0)],
        ["false_positive", "false_negative"], default="correct"
    )
    out["model"] = model_kind
    out["seed"] = seed
    return out


## 9. Smoke test and full factorial execution

The notebook refuses to produce attribution tables from partial results. A dry run checks data and intervention integrity without downloading or fine-tuning the encoder. The full run requires 4 factorial cells × 6 models × 5 seeds = **120 complete runs**. If a run fails, the failure is recorded and the final analysis stops rather than silently dropping the cell.

In [26]:
# 9.1 Smoke test: no model training, but all design invariants are checked.
SMOKE_ROWS = []
for c, p in itertools.product([0, 1], repeat=2):
    cell = build_factorial_cell(c, p, CFG.SEEDS[0])
    SMOKE_ROWS.append({
        "content_on": c, "protocol_on": p,
        "train_n": len(cell["train"]), "adapt_n": len(cell["adapt"]), "test_n": len(cell["test"]),
        "train_true_prior": cell["train"]["true_label"].mean(),
        "train_observed_prior": cell["train"]["observed_label"].mean(),
        "test_true_prior": cell["test"]["true_label"].mean(),
    })
SMOKE_DF = pd.DataFrame(SMOKE_ROWS)
assert SMOKE_DF[["train_n", "adapt_n", "test_n"]].nunique().max() == 1
assert np.isclose(SMOKE_DF["test_true_prior"].iloc[0], SMOKE_DF["test_true_prior"]).all()
print("Smoke test passed.")
display(SMOKE_DF)

Smoke test passed.


,content_on,protocol_on,train_n,adapt_n,test_n,train_true_prior,train_observed_prior,test_true_prior
0,0,0,37390,6798,10197,0.515485,0.515485,0.515544
1,0,1,37390,6798,10197,0.515485,0.459160,0.515544
2,1,0,37390,6798,10197,0.515485,0.515485,0.515544
3,1,1,37390,6798,10197,0.515485,0.459160,0.515544


In [27]:
# # 9.2 Full factorial runner
# from tqdm.auto import tqdm

# CFG.RUN_TRAINING = True
# if CFG.RUN_TRAINING and any(m in {"ERM", "DANN", "CORAL", "PROTOCOL_CORRECTED"} for m in CFG.MODELS) and not torch.cuda.is_available():
#     raise RuntimeError("CUDA is required for the 80 neural runs; this runtime has no GPU. Run the notebook on a GPU-enabled environment.")

# EXPECTED_KEYS = [
#     {"model": m, "seed": s, "content_on": c, "protocol_on": p}
#     for m, s, (c, p) in itertools.product(
#         CFG.MODELS,
#         CFG.SEEDS,
#         itertools.product([0, 1], repeat=2)
#     )
# ]

# # --- resume support: load already-completed keys so a restarted kernel
# # does not retrain runs that already succeeded. ---
# RESULTS_PATH = OUT_ROOT / "tables" / "factorial_results.csv"
# DONE_KEYS = set()
# if RESULTS_PATH.exists():
#     _done_df = pd.read_csv(RESULTS_PATH)
#     if len(_done_df):
#         DONE_KEYS = set(
#             zip(_done_df["model"], _done_df["seed"].astype(int),
#                 _done_df["content_on"].astype(int), _done_df["protocol_on"].astype(int))
#         )
# print(f"{len(DONE_KEYS)} / {len(EXPECTED_KEYS)} runs already completed; resuming the rest.")

# if CFG.RUN_TRAINING:
#     failures = []

#     pending_keys = [
#         key for key in EXPECTED_KEYS
#         if (key["model"], key["seed"], key["content_on"], key["protocol_on"]) not in DONE_KEYS
#     ]

#     for key in tqdm(
#         pending_keys,
#         total=len(pending_keys),
#         desc="Full factorial training",
#         unit="run"
#     ):
#         try:
#             cell = build_factorial_cell(
#                 key["content_on"],
#                 key["protocol_on"],
#                 key["seed"]
#             )

#             train_one_run(key["model"], cell, key["seed"])

#             print("completed", key)

#         except Exception as exc:
#             failure = {"run_id": RUN_ID, **key, "error": repr(exc)}
#             failures.append(failure)

#             append_deduplicated(
#                 pd.DataFrame([failure]),
#                 OUT_ROOT / "tables" / "factorial_failures.csv",
#                 ["run_id", "model", "seed", "content_on", "protocol_on"]
#             )

#             print("FAILED", failure)

#     if failures:
#         print(
#             f"{len(failures)} runs failed; "
#             "downstream analysis is intentionally blocked."
#         )
# else:
#     print(
#         "DRY RUN: set CFG.RUN_TRAINING=True "
#         "to execute the declared full factorial."
#     )

In [28]:
# # 9.2 Full factorial runner (Colab slice: PROTOCOL_CORRECTED block only)
# from tqdm.auto import tqdm

# CFG.RUN_TRAINING = True
# if CFG.RUN_TRAINING and any(m in {"ERM", "DANN", "CORAL", "PROTOCOL_CORRECTED"} for m in CFG.MODELS) and not torch.cuda.is_available():
#     raise RuntimeError("CUDA is required for the 80 neural runs; this runtime has no GPU. Run the notebook on a GPU-enabled environment.")

# EXPECTED_KEYS = [
#     {"model": m, "seed": s, "content_on": c, "protocol_on": p}
#     for m, s, (c, p) in itertools.product(
#         CFG.MODELS,
#         CFG.SEEDS,
#         itertools.product([0, 1], repeat=2)
#     )
# ]

# RESULTS_PATH = OUT_ROOT / "tables" / "factorial_results.csv"
# DONE_KEYS = set()
# if RESULTS_PATH.exists():
#     _done_df = pd.read_csv(RESULTS_PATH)
#     if len(_done_df):
#         DONE_KEYS = set(
#             zip(_done_df["model"], _done_df["seed"].astype(int),
#                 _done_df["content_on"].astype(int), _done_df["protocol_on"].astype(int))
#         )
# print(f"{len(DONE_KEYS)} / {len(EXPECTED_KEYS)} runs already completed; resuming the rest.")

# # --- only this block is new: restrict to a specific index range ---
# START_IDX, END_IDX = 52, 57   # inclusive-exclusive slice -> PROTOCOL_CORRECTED (indices 100–119)
# SLICE_KEYS = EXPECTED_KEYS[START_IDX:END_IDX]
# print(f"Colab will handle indices {START_IDX}:{END_IDX} -> {len(SLICE_KEYS)} keys "
#       f"(models: {sorted(set(k['model'] for k in SLICE_KEYS))})")

# if CFG.RUN_TRAINING:
#     failures = []

#     pending_keys = [
#         key for key in SLICE_KEYS
#         if (key["model"], key["seed"], key["content_on"], key["protocol_on"]) not in DONE_KEYS
#     ]

#     for key in tqdm(pending_keys, total=len(pending_keys), desc="Full factorial training (Colab slice)", unit="run"):
#         try:
#             cell = build_factorial_cell(key["content_on"], key["protocol_on"], key["seed"])
#             train_one_run(key["model"], cell, key["seed"])
#             print("completed", key)
#         except Exception as exc:
#             failure = {"run_id": RUN_ID, **key, "error": repr(exc)}
#             failures.append(failure)
#             append_deduplicated(
#                 pd.DataFrame([failure]),
#                 OUT_ROOT / "tables" / "factorial_failures.csv",
#                 ["run_id", "model", "seed", "content_on", "protocol_on"]
#             )
#             print("FAILED", failure)

#     if failures:
#         print(f"{len(failures)} runs failed; downstream analysis is intentionally blocked.")

In [29]:
# 9.2 Full factorial runner (Colab slice: PROTOCOL_CORRECTED block only)
from tqdm.auto import tqdm

CFG.RUN_TRAINING = True
if CFG.RUN_TRAINING and any(m in {"ERM", "DANN", "CORAL", "PROTOCOL_CORRECTED"} for m in CFG.MODELS) and not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for the 80 neural runs; this runtime has no GPU. Run the notebook on a GPU-enabled environment.")

EXPECTED_KEYS = [
    {"model": m, "seed": s, "content_on": c, "protocol_on": p}
    for m, s, (c, p) in itertools.product(
        CFG.MODELS,
        CFG.SEEDS,
        itertools.product([0, 1], repeat=2)
    )
]

RESULTS_PATH = OUT_ROOT / "tables" / "factorial_results.csv"
DONE_KEYS = set()
if RESULTS_PATH.exists():
    _done_df = pd.read_csv(RESULTS_PATH)
    if len(_done_df):
        DONE_KEYS = set(
            zip(_done_df["model"], _done_df["seed"].astype(int),
                _done_df["content_on"].astype(int), _done_df["protocol_on"].astype(int))
        )
print(f"{len(DONE_KEYS)} / {len(EXPECTED_KEYS)} runs already completed; resuming the rest.")

# --- only this block is new: restrict to a specific index range ---
START_IDX, END_IDX = 107, 121   # inclusive-exclusive slice -> PROTOCOL_CORRECTED (indices 100–119)
SLICE_KEYS = EXPECTED_KEYS[START_IDX:END_IDX]
print(f"Colab will handle indices {START_IDX}:{END_IDX} -> {len(SLICE_KEYS)} keys "
      f"(models: {sorted(set(k['model'] for k in SLICE_KEYS))})")

if CFG.RUN_TRAINING:
    failures = []

    pending_keys = [
        key for key in SLICE_KEYS
        if (key["model"], key["seed"], key["content_on"], key["protocol_on"]) not in DONE_KEYS
    ]

    for key in tqdm(pending_keys, total=len(pending_keys), desc="Full factorial training (Colab slice)", unit="run"):
        try:
            cell = build_factorial_cell(key["content_on"], key["protocol_on"], key["seed"])
            train_one_run(key["model"], cell, key["seed"])
            print("completed", key)
        except Exception as exc:
            failure = {"run_id": RUN_ID, **key, "error": repr(exc)}
            failures.append(failure)
            append_deduplicated(
                pd.DataFrame([failure]),
                OUT_ROOT / "tables" / "factorial_failures.csv",
                ["run_id", "model", "seed", "content_on", "protocol_on"]
            )
            print("FAILED", failure)

    if failures:
        print(f"{len(failures)} runs failed; downstream analysis is intentionally blocked.")

0 / 120 runs already completed; resuming the rest.
Colab will handle indices 107:121 -> 13 keys (models: ['PROTOCOL_CORRECTED'])


Full factorial training (Colab slice):   0%|          | 0/13 [00:00<?, ?run/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


completed {'model': 'PROTOCOL_CORRECTED', 'seed': 9001, 'content_on': 1, 'protocol_on': 0}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


completed {'model': 'PROTOCOL_CORRECTED', 'seed': 9001, 'content_on': 1, 'protocol_on': 1}


In [30]:
import zipfile
import os

STAGE = "107-121"
zip_path = f"/kaggle/working/{STAGE}_backup.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_STORED) as z:
    for root, dirs, files in os.walk("/kaggle/working/results"):
        for file in files:
            fp = os.path.join(root, file)
            arc = os.path.relpath(fp, "/kaggle/working")
            z.write(fp, arc)

print("Backup Saved:", zip_path)
print("ZIP Size:", os.path.getsize(zip_path) / (1024**3), "GiB")

KeyboardInterrupt: 

In [31]:
import os
os.remove('/kaggle/working/107-121_backup.zip')

In [ ]:
# import zipfile
# import os

# STAGE = "107-121"
# zip_path = f"/kaggle/working/{STAGE}_backup.zip"

# with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
#     for root, dirs, files in os.walk("/kaggle/working/results"):
#         for file in files:
#             fp = os.path.join(root, file)
#             arc = os.path.relpath(fp, "/kaggle/working")
#             z.write(fp, arc)

# print("Backup Saved:", zip_path)
# print("ZIP Size:", os.path.getsize(zip_path) / (1024**3), "GiB")

In [32]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ['KAGGLE_USERNAME'] = user_secrets.get_secret("KAGGLE_USERNAME")
os.environ['KAGGLE_KEY'] = user_secrets.get_secret("KAGGLE_KEY")

In [ ]:
# import json, os


# os.makedirs('/kaggle/working/dataset_meta_v2', exist_ok=True)
# metadata = {
#     "title": "107-121-backup-v2",
#     "id": "slytherin401533/107-121-backup-v2",
#     "licenses": [{"name": "CC0-1.0"}]
# }
# with open('/kaggle/working/dataset_meta_v2/dataset-metadata.json', 'w') as f:
#     json.dump(metadata, f)

# !ln -sf /kaggle/working/107-121_backup.zip /kaggle/working/dataset_meta_v2/107-121_backup.zip

# !kaggle datasets create -p /kaggle/working/dataset_meta_v2 --dir-mode zip

In [ ]:
import json, os

result_folder = '/kaggle/working/results'   
metadata = {
    "title": "107-121-backup-v2",
    "id": "slytherin401533/107-121-backup-v2",
    "licenses": [{"name": "CC0-1.0"}]
}
with open(os.path.join(result_folder, 'dataset-metadata.json'), 'w') as f:
    json.dump(metadata, f)

!kaggle datasets create -p {result_folder} --dir-mode zip

Starting upload for file manifests.zip
100%|████████████████████████████████████████| 889k/889k [00:00<00:00, 2.14MB/s]
Upload successful: manifests.zip (889KB)
Starting upload for file figures.zip
100%|█████████████████████████████████████████| 22.0/22.0 [00:00<00:00, 54.7B/s]
Upload successful: figures.zip (22B)
Starting upload for file tables.zip
100%|██████████████████████████████████████| 2.04k/2.04k [00:00<00:00, 5.03kB/s]
Upload successful: tables.zip (2KB)


In [ ]:
import os

checkpoint_dir = "/kaggle/working/results/checkpoints"

files = []
for root, dirs, filenames in os.walk(checkpoint_dir):
    for f in filenames:
        files.append(os.path.join(root, f))

print("Checkpoint files:", len(files))

for f in sorted(files):
    print(os.path.relpath(f, checkpoint_dir))

In [ ]:
import zipfile

zip_path = "/kaggle/working/41-51_backup.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    checkpoint_files = [
        f for f in z.namelist()
        if f.startswith("results/checkpoints/")
    ]

print("Checkpoint files in ZIP:", len(checkpoint_files))

for f in checkpoint_files:
    print(f)

In [ ]:
# 9.3 TACL-safe factorial loading and completeness gate
def require_complete_factorial(results_path=None):
    results_path = Path(results_path or (OUT_ROOT / "tables" / "factorial_results.csv"))
    if not results_path.exists():
        raise RuntimeError("No factorial results found. Complete all declared runs first.")
    df = pd.read_csv(results_path)
    keys = ["model", "seed", "content_on", "protocol_on"]
    if "run_id" not in df.columns:
        raise RuntimeError("factorial_results.csv has no run_id provenance column.")
    df = df[df["run_id"].astype(str) == str(RUN_ID)].copy()
    if df.empty:
        raise RuntimeError(f"No rows match current RUN_ID={RUN_ID}.")
    required = pd.DataFrame(EXPECTED_KEYS)
    missing = required.merge(df[keys].drop_duplicates(), on=keys, how="left", indicator=True)
    missing = missing[missing["_merge"] == "left_only"].drop(columns="_merge")
    if len(missing):
        raise RuntimeError(f"Incomplete factorial: {len(missing)} missing runs.\n{missing.head(10)}")
    if df[keys].duplicated().any():
        raise RuntimeError("Duplicate factorial keys detected.")
    if len(df) != len(EXPECTED_KEYS):
        raise RuntimeError(f"Expected {len(EXPECTED_KEYS)} rows, observed {len(df)}.")
    return df

RESULTS_PATH = OUT_ROOT / "tables" / "factorial_results.csv"
if RESULTS_PATH.exists():
    FACTORIAL_RESULTS = require_complete_factorial(RESULTS_PATH)
else:
    FACTORIAL_RESULTS = None
    print("No complete current-run factorial loaded; downstream analyses remain gated.")


### 9.4 Figure -- model-panel Macro-F1 comparison

One bar per completed model, mean Macro-F1 over all completed cells and seeds, with a 95% seed interval. Renders with whatever subset of the six models is complete at run time -- two bars today, six once the neural arms finish.

In [ ]:
# 9.4 Figure -- full model-panel Macro-F1 comparison
if FACTORIAL_RESULTS is not None:
    order = [m for m in ["MAJORITY","TFIDF_LOGREG","ERM","DANN","CORAL","PROTOCOL_CORRECTED"]
             if m in FACTORIAL_RESULTS["model"].unique()]
    fig, ax = plt.subplots(figsize=TACL_2COL)
    sns.barplot(data=FACTORIAL_RESULTS, x="model", y="macro_f1", order=order,
                errorbar=("pi", 95), capsize=0.12, err_kws={"linewidth": 1.1, "color": "#333333"},
                palette=TACL_PALETTE[:len(order)], ax=ax)
    ax.set_xlabel(""); ax.set_ylabel("Macro-F1")
    ax.set_title("Model panel: completed factorial coverage", loc="left", fontsize=10)
    ax.set_ylim(0, 1.0)
    for lbl in ax.get_xticklabels():
        lbl.set_rotation(15); lbl.set_ha("right")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_model_panel_macro_f1.pdf")
    fig.savefig(FIG_DIR / "fig_model_panel_macro_f1.png")
    plt.show()
else:
    print("Model-panel figure will render once FACTORIAL_RESULTS is populated by a completed run.")


## 10. Exact Shapley attribution and uncertainty

For each model and seed, let `V(c,p)` be the clean-test macro-F1 in the cell defined by content state `c` and protocol state `p`. Define the performance drop relative to the no-shift cell as `D(c,p) = V(0,0) − V(c,p)`. The exact two-factor Shapley contributions are:

\[
\phi_C = \frac{1}{2}\{D(1,0)-D(0,0) + D(1,1)-D(0,1)\},
\]
\[
\phi_P = \frac{1}{2}\{D(0,1)-D(0,0) + D(1,1)-D(1,0)\}.
\]

The decomposition is calculated independently for every seed. Confidence intervals are bootstrap intervals over seed-level contributions; they quantify run-to-run uncertainty, not population uncertainty over all possible corpora.

In [ ]:
# 10.1 Attribution, interactions, paired randomization, and confidence intervals

def paired_randomization_p(diffs):
    diffs = np.asarray(diffs, dtype=float)
    diffs = diffs[np.isfinite(diffs)]
    if len(diffs) == 0: return np.nan
    observed = abs(diffs.mean())
    vals = []
    for signs in itertools.product([-1.0, 1.0], repeat=len(diffs)):
        vals.append(abs((diffs * np.asarray(signs)).mean()))
    return float((1 + sum(v >= observed - 1e-12 for v in vals)) / (1 + len(vals)))

def bootstrap_ci(values, seed=7, n_boot=None):
    x = np.asarray(values, dtype=float); x = x[np.isfinite(x)]
    if len(x) == 0: return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boot = np.array([rng.choice(x, size=len(x), replace=True).mean() for _ in range(n_boot or CFG.N_BOOTSTRAP)])
    return float(x.mean()), float(np.quantile(boot, 0.025)), float(np.quantile(boot, 0.975))

def shapley_two_factor(group):
    vals = {(int(r.content_on), int(r.protocol_on)): float(r.macro_f1) for r in group.itertuples()}
    needed = {(0,0), (1,0), (0,1), (1,1)}
    if set(vals) != needed: raise RuntimeError(f"Missing factorial values: {set(vals)}")
    v00, v10, v01, v11 = vals[(0,0)], vals[(1,0)], vals[(0,1)], vals[(1,1)]
    d10, d01, d11 = v00 - v10, v00 - v01, v00 - v11
    phi_c = 0.5 * (d10 + (d11 - d01))
    phi_p = 0.5 * (d01 + (d11 - d10))
    interaction = d11 - d10 - d01
    return {"phi_content": phi_c, "phi_protocol": phi_p, "total_drop": d11, "interaction": interaction,
            "efficiency_error": (phi_c + phi_p) - d11}

def attribution_table(df):
    rows = []
    for (model, seed), g in df.groupby(["model", "seed"]):
        out = shapley_two_factor(g)
        rows.append({"model": model, "seed": seed, **out})
    return pd.DataFrame(rows)

def summarize_attribution(attr):
    rows = []
    for model, g in attr.groupby("model"):
        for metric in ["phi_content", "phi_protocol", "total_drop", "interaction"]:
            mean, lo, hi = bootstrap_ci(g[metric].to_numpy())
            rows.append({"model": model, "metric": metric, "mean": mean, "ci_low": lo, "ci_high": hi, "n_seeds": len(g)})
    return pd.DataFrame(rows)

if FACTORIAL_RESULTS is not None:
    ATTRIBUTION_DF = attribution_table(FACTORIAL_RESULTS)
    ATTRIBUTION_SUMMARY = summarize_attribution(ATTRIBUTION_DF)
    display(ATTRIBUTION_SUMMARY)
    ATTRIBUTION_DF.to_csv(OUT_ROOT / "tables" / "shapley_seed_level.csv", index=False)
    ATTRIBUTION_SUMMARY.to_csv(OUT_ROOT / "tables" / "shapley_summary.csv", index=False)
else:
    print("Attribution blocked until the complete factorial exists.")

### 10.5 Figure -- calibration reliability diagram

Uses `reload_predictions` (Section 8.5) on the no-shift `(0,0)` cell, first seed, so the diagram reflects one clean, unshifted evaluation per model rather than a mixture of intervention conditions.

In [ ]:
# 10.5 Figure -- calibration reliability diagram (no-shift cell)
def reliability_curve(y_true, prob_pos, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    xs, ys = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        mask = (prob_pos >= lo) & (prob_pos < hi if hi < 1 else prob_pos <= hi)
        if mask.sum() > 0:
            xs.append(prob_pos[mask].mean()); ys.append(y_true[mask].mean())
    return np.array(xs), np.array(ys)

if FACTORIAL_RESULTS is not None:
    fig, ax = plt.subplots(figsize=TACL_1COL)
    ax.plot([0, 1], [0, 1], color="#999999", lw=0.8, ls="--", label="Perfect calibration")
    models_present = list(FACTORIAL_RESULTS["model"].unique())
    colors = dict(zip(models_present, TACL_PALETTE))
    plotted = False
    for model_kind in models_present:
        et = reload_predictions(model_kind, CFG.SEEDS[0], 0, 0)
        if et is None:
            continue
        xs, ys = reliability_curve(et["y_true"].to_numpy(), et["prob_pos"].to_numpy())
        ax.plot(xs, ys, marker="o", ms=3, color=colors[model_kind], label=model_kind)
        plotted = True
    if plotted:
        ax.set_xlabel("Predicted probability"); ax.set_ylabel("Observed frequency")
        ax.set_title("Calibration reliability, no-shift cell", loc="left", fontsize=10)
        ax.legend(frameon=False, fontsize=7)
        fig.tight_layout()
        fig.savefig(FIG_DIR / "fig_calibration_reliability.pdf")
        fig.savefig(FIG_DIR / "fig_calibration_reliability.png")
        plt.show()
    else:
        plt.close(fig)
        print("No reloadable classical checkpoints found yet for calibration figure.")
else:
    print("Calibration figure pending a completed factorial.")


In [ ]:
# 10.2 Paired tests with Holm correction and protocol selectivity
def holm_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    out = np.full(len(p), np.nan)
    finite = np.where(np.isfinite(p))[0]
    order = finite[np.argsort(p[finite])]
    running, m = 0.0, len(order)
    for rank, idx in enumerate(order):
        running = max(running, min(1.0, (m - rank) * p[idx]))
        out[idx] = running
    return out

if FACTORIAL_RESULTS is not None:
    test_rows = []
    for model, g in FACTORIAL_RESULTS.groupby("model"):
        pivot = g.pivot_table(index="seed", columns=["content_on", "protocol_on"], values="macro_f1")
        for contrast, a, b in [("content", (0,0), (1,0)), ("content_at_protocol_on", (0,1), (1,1)),
                               ("protocol", (0,0), (0,1)), ("protocol_at_content_on", (1,0), (1,1))]:
            if a not in pivot.columns or b not in pivot.columns:
                continue
            diffs = (pivot[a] - pivot[b]).dropna()
            test_rows.append({"model": model, "contrast": contrast, "mean_drop": float(diffs.mean()),
                              "p_randomization": paired_randomization_p(diffs.to_numpy()), "n_pairs": int(len(diffs))})
    PAIRED_TESTS = pd.DataFrame(test_rows)
    if len(PAIRED_TESTS):
        PAIRED_TESTS["p_holm"] = holm_adjust(PAIRED_TESTS["p_randomization"].to_numpy())
    display(PAIRED_TESTS)
    PAIRED_TESTS.to_csv(OUT_ROOT / "tables" / "paired_randomization_tests.csv", index=False)

    SELECTIVITY = pd.DataFrame()
    models = set(FACTORIAL_RESULTS["model"].unique())
    if {"ERM", "PROTOCOL_CORRECTED"}.issubset(models):
        er = FACTORIAL_RESULTS[FACTORIAL_RESULTS.model == "ERM"].set_index(["seed", "content_on", "protocol_on"])["macro_f1"]
        pc = FACTORIAL_RESULTS[FACTORIAL_RESULTS.model == "PROTOCOL_CORRECTED"].set_index(["seed", "content_on", "protocol_on"])["macro_f1"]
        joined = pd.concat([er.rename("erm"), pc.rename("protocol_corrected")], axis=1).dropna().reset_index()
        joined["gain"] = joined["protocol_corrected"] - joined["erm"]
        SELECTIVITY = joined.groupby("protocol_on")["gain"].agg(["mean", "std", "count"]).reset_index()
        display(SELECTIVITY)
        SELECTIVITY.to_csv(OUT_ROOT / "tables" / "protocol_correction_selectivity.csv", index=False)
else:
    PAIRED_TESTS = None
    SELECTIVITY = None


### 10.6 Correction efficiency across factors and methods

$CE_{i,m} = R_{i,m} / \varphi_i$, where $\varphi_i$ is ERM's Shapley attribution for factor $i$ and $R_{i,m}$ is the mean Macro-F1 gain of correction method $m$ over ERM, restricted to the cells where factor $i$ is active. $CE \approx 1$ means the method recovers essentially all of the ERM-attributed drop for that factor; $CE \approx 0$ means it recovers none of it.

In [ ]:
# 10.6 Correction efficiency across factors and methods
if FACTORIAL_RESULTS is not None and "ERM" in FACTORIAL_RESULTS["model"].unique():
    erm_phi_rows = ATTRIBUTION_SUMMARY[(ATTRIBUTION_SUMMARY.model == "ERM") &
                                        (ATTRIBUTION_SUMMARY.metric.isin(["phi_content", "phi_protocol"]))]
    phi_lookup = {row.metric.replace("phi_", ""): row.__getattribute__("mean") for row in erm_phi_rows.itertuples()}

    ce_rows = []
    erm_series = FACTORIAL_RESULTS[FACTORIAL_RESULTS.model == "ERM"].set_index(["seed", "content_on", "protocol_on"])["macro_f1"]
    for method in ["DANN", "CORAL", "PROTOCOL_CORRECTED"]:
        if method not in FACTORIAL_RESULTS["model"].unique():
            continue
        corrected_series = FACTORIAL_RESULTS[FACTORIAL_RESULTS.model == method].set_index(["seed", "content_on", "protocol_on"])["macro_f1"]
        joined = pd.concat([erm_series.rename("erm"), corrected_series.rename("corrected")], axis=1).dropna().reset_index()
        joined["gain"] = joined["corrected"] - joined["erm"]
        for factor, mask in [("content", joined.content_on == 1), ("protocol", joined.protocol_on == 1)]:
            R = joined.loc[mask, "gain"].mean()
            phi = phi_lookup.get(factor, np.nan)
            ce_rows.append({"method": method, "factor": factor, "R": R, "phi_ERM": phi,
                             "CE": (R / phi) if phi not in (0, np.nan) and not pd.isna(phi) else np.nan})
    CORRECTION_EFFICIENCY = pd.DataFrame(ce_rows)
    if len(CORRECTION_EFFICIENCY):
        display(CORRECTION_EFFICIENCY)
        CORRECTION_EFFICIENCY.to_csv(OUT_ROOT / "tables" / "correction_efficiency.csv", index=False)
else:
    CORRECTION_EFFICIENCY = None
    print("Correction efficiency requires the completed ERM plus at least one correction-method arm.")


In [ ]:
# 10.7 Figure -- correction efficiency by method and factor
if CORRECTION_EFFICIENCY is not None and len(CORRECTION_EFFICIENCY):
    fig, ax = plt.subplots(figsize=TACL_2COL)
    sns.barplot(data=CORRECTION_EFFICIENCY, x="method", y="CE", hue="factor",
                palette=[TACL_PALETTE[0], TACL_PALETTE[1]], ax=ax)
    ax.axhline(0, color="#333333", lw=0.8)
    ax.axhline(1, color="#999999", lw=0.8, ls="--")
    ax.set_ylabel(r"Correction efficiency $CE_{i,m}$"); ax.set_xlabel("")
    ax.set_title("Selective recovery of Shapley-attributed drop", loc="left", fontsize=10)
    ax.legend(title="Factor", frameon=False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_correction_efficiency.pdf")
    fig.savefig(FIG_DIR / "fig_correction_efficiency.png")
    plt.show()
else:
    print("Correction-efficiency figure is pending the neural correction arms (DANN / CORAL / PROTOCOL_CORRECTED).")


In [ ]:
# 10.3 Factorial plots
if FACTORIAL_RESULTS is not None:
    plt.figure(figsize=(8, 5))
    sns.pointplot(data=FACTORIAL_RESULTS, x="content_on", y="macro_f1", hue="protocol_on", errorbar="sd", dodge=0.2)
    plt.xlabel("Content intervention (0=off, 1=on)"); plt.ylabel("Macro-F1")
    plt.title("Complete 2×2 intervention factorial")
    plt.tight_layout(); plt.savefig(OUT_ROOT / "figures" / "factorial_macro_f1.png", dpi=220); plt.show()

    plot_df = ATTRIBUTION_SUMMARY[ATTRIBUTION_SUMMARY.metric.isin(["phi_content", "phi_protocol"])]
    plt.figure(figsize=(8, 5))
    sns.barplot(data=plot_df, x="model", y="mean", hue="metric", errorbar=None)
    plt.axhline(0, color="black", lw=0.8)
    plt.ylabel("Seed-level Shapley contribution to macro-F1 drop")
    plt.title("Intervention-defined shift attribution")
    plt.tight_layout(); plt.savefig(OUT_ROOT / "figures" / "shapley_contributions.png", dpi=220); plt.show()

## 11. Robustness to intervention strength

A 2×2 result at one arbitrary intervention strength is not sufficient. The final paper must include a held-out-strength robustness arm. The following cell defines the grid; it is not included in the primary Shapley estimate. The robustness arm must use the same seeds, source-validation rule, and failure gate.

In [ ]:
# 11.1 Robustness grid definition and optional runner
ROBUSTNESS_GRID = pd.DataFrame([
    {"content_strength": c, "protocol_noise_scale": p}
    for c, p in itertools.product([1.0, 2.0, 3.0], [0.5, 1.0, 1.5])
])
# The primary notebook keeps the transition matrix fixed. For the robustness arm,
# scale off-diagonal noise while renormalizing each row.
def scaled_transition(scale):
    T = CFG.PROTOCOL_TRANSITION.copy()
    base = np.eye(2)
    out = base + scale * (T - base)
    out = out / out.sum(axis=1, keepdims=True)
    return out

def robustness_spec():
    specs = []
    for row in ROBUSTNESS_GRID.itertuples():
        specs.append({"content_strength": row.content_strength, "transition": scaled_transition(row.protocol_noise_scale)})
    return specs
print(ROBUSTNESS_GRID)
if CFG.RUN_ROBUSTNESS:
    print("Robustness runner should be executed only after the primary 120-run factorial is complete.")

### 11.2 Robustness runner

For each grid point, this cell rebuilds all four factorial cells at that content strength / protocol noise scale and computes the two-factor Shapley attribution. Currently runs the classical arm (`TFIDF_LOGREG`) only; extend the inner loop to the neural model kinds once GPU capacity allows the full robustness sweep.

In [ ]:
# 11.2 Robustness runner with validation-consistent protocol intervention
def protocol_shift_at(df, on, seed, transition):
    out = df.copy().reset_index(drop=True)
    out["observed_label"] = out["true_label"].astype(float)
    if not on:
        return out
    rng = np.random.default_rng(seed)
    y = out["true_label"].astype(int).to_numpy()
    out["observed_label"] = np.asarray([rng.choice([0, 1], p=transition[yi]) for yi in y], dtype=float)
    return out

def build_cell_at_strength(content_on, protocol_on, seed, content_strength, transition):
    train = protocol_shift_at(ANCHOR_SPLITS["train"], protocol_on, seed + 101, transition)
    val = protocol_shift_at(ANCHOR_SPLITS["val"], protocol_on, seed + 102, transition)
    adapt, test = content_shift_pools(ANCHOR_SPLITS["adapt"], ANCHOR_SPLITS["test"], ANCHOR_SPLITS["reserve"], content_on, seed + 103, strength=content_strength)
    train["observed_label"] = train["observed_label"].astype(int)
    val["observed_label"] = val["observed_label"].astype(int)
    return {"train": train, "val": val, "adapt": adapt, "test": test, "content_on": int(content_on), "protocol_on": int(protocol_on), "seed": int(seed)}

if CFG.RUN_ROBUSTNESS:
    rob_rows = []
    for row in ROBUSTNESS_GRID.itertuples():
        transition = scaled_transition(row.protocol_noise_scale)
        for seed in CFG.SEEDS:
            values = {}
            for c, p in itertools.product([0, 1], repeat=2):
                cell = build_cell_at_strength(c, p, seed, row.content_strength, transition)
                for model_kind in CFG.ROBUSTNESS_MODELS:
                    result = train_one_run(model_kind, cell, seed, persist=False,
                                           run_name=f"ROB_{model_kind}_cs{row.content_strength}_pn{row.protocol_noise_scale}_c{c}p{p}_s{seed}")
                    values.setdefault(model_kind, {})[(c, p)] = result["macro_f1"]
            for model_kind, v in values.items():
                d10, d01, d11 = v[(0,0)] - v[(1,0)], v[(0,0)] - v[(0,1)], v[(0,0)] - v[(1,1)]
                rob_rows.append({"content_strength": row.content_strength, "protocol_noise_scale": row.protocol_noise_scale,
                                 "model": model_kind, "seed": seed, "phi_content": 0.5 * (d10 + d11 - d01),
                                 "phi_protocol": 0.5 * (d01 + d11 - d10)})
    ROBUSTNESS_RESULTS = pd.DataFrame(rob_rows)
    ROBUSTNESS_RESULTS.to_csv(OUT_ROOT / "tables" / "robustness_results.csv", index=False)
    display(ROBUSTNESS_RESULTS.groupby(["model", "content_strength", "protocol_noise_scale"])[["phi_content", "phi_protocol"]].mean())
else:
    ROBUSTNESS_RESULTS = None
    print("Robustness disabled; do not make robustness claims.")


In [ ]:
# 11.3 Figure -- attribution stability across intervention strength
if ROBUSTNESS_RESULTS is not None and len(ROBUSTNESS_RESULTS):
    summary = ROBUSTNESS_RESULTS.groupby("protocol_noise_scale")[["phi_content", "phi_protocol"]].mean().reset_index()
    fig, ax = plt.subplots(figsize=TACL_1COL)
    ax.plot(summary.protocol_noise_scale, summary.phi_content, marker="o", color=TACL_PALETTE[0], label=r"$\phi_{content}$")
    ax.plot(summary.protocol_noise_scale, summary.phi_protocol, marker="s", color=TACL_PALETTE[1], label=r"$\phi_{protocol}$")
    ax.set_xlabel("Protocol noise scale"); ax.set_ylabel("Mean Shapley contribution")
    ax.set_title("Robustness to intervention strength", loc="left", fontsize=10)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_robustness_curve.pdf")
    fig.savefig(FIG_DIR / "fig_robustness_curve.png")
    plt.show()
else:
    print("Robustness-curve figure is pending CFG.RUN_ROBUSTNESS=True execution.")


## 12. External real-corpus stress test

Real corpora are not assigned synthetic factor labels. For each external target, the model receives only target adaptation text without labels, and the target test labels are used once for final evaluation. Results include corpus provenance, class-label coverage, text/register signatures, and an OOD flag relative to the synthetic anchor range.

This section answers: **Does the model fail on real targets, and are their observable signatures outside the synthetic support?** It does not answer: **What causal percentage of the real drop is platform or language?**

In [ ]:
# 12.1 Observable shift signatures

def script_features(texts):
    texts = pd.Series(texts).astype(str)
    chars = texts.str.len().sum()
    bangla = texts.map(lambda s: sum("\u0980" <= ch <= "\u09ff" for ch in s)).sum()
    latin = texts.map(lambda s: sum(("a" <= ch.lower() <= "z") for ch in s)).sum()
    return {
        "mean_chars": float(texts.str.len().mean()),
        "mean_tokens": float(texts.str.split().str.len().mean()),
        "bangla_char_fraction": float(bangla / max(chars, 1)),
        "latin_char_fraction": float(latin / max(chars, 1)),
    }

def js_divergence(p, q):
    keys = sorted(set(p) | set(q))
    p = np.array([p.get(k, 0.0) for k in keys], dtype=float); q = np.array([q.get(k, 0.0) for k in keys], dtype=float)
    p = p / max(p.sum(), 1e-12); q = q / max(q.sum(), 1e-12); m = 0.5 * (p + q)
    return float(0.5 * (stats.entropy(p, m) + stats.entropy(q, m)))

def signature(anchor_df, target_df):
    out = script_features(target_df["text"])
    out.update({f"anchor_{k}": v for k, v in script_features(anchor_df["text"]).items()})
    if "topic" in anchor_df and "topic" in target_df:
        p = anchor_df["topic"].fillna("__unknown__").value_counts(normalize=True).to_dict()
        q = target_df["topic"].fillna("__unknown__").value_counts(normalize=True).to_dict()
        out["topic_js"] = js_divergence(p, q)
    if target_df["true_label"].notna().any(): out["target_positive_rate"] = float(target_df["true_label"].mean())
    else: out["target_positive_rate"] = np.nan
    return out

REAL_SIGNATURES = []
for source, parts in REAL_TARGET_SPLITS.items():
    REAL_SIGNATURES.append({"source": source, **signature(ANCHOR_SPLITS["train"], parts["test"])})
REAL_SIGNATURES_DF = pd.DataFrame(REAL_SIGNATURES)
display(REAL_SIGNATURES_DF)
REAL_SIGNATURES_DF.to_csv(OUT_ROOT / "tables" / "real_target_signatures.csv", index=False)

In [ ]:
# 12.2 Optional real-target training/evaluation

def train_real_target(source, model_kind, seed):
    parts = REAL_TARGET_SPLITS[source]
    if not parts["test"]["true_label"].notna().all():
        return {"source": source, "model": model_kind, "seed": seed, "status": "NO_RELIABLE_TEST_LABELS"}
    cell = {
        "train": protocol_shift(ANCHOR_SPLITS["train"], False, seed),
        "val": protocol_shift(ANCHOR_SPLITS["val"], False, seed + 1),
        "adapt": parts["adapt"].assign(observed_label=-1),
        "test": parts["test"], "content_on": np.nan, "protocol_on": np.nan, "seed": seed,
    }
    out = train_one_run(model_kind, cell, seed, run_name=f"REAL_{source}_{model_kind}_s{seed}", persist=False)
    out["source"] = source; out["status"] = "OK"
    return out

if CFG.RUN_REAL_EVAL:
    real_rows = []
    for source in REAL_TARGET_SPLITS:
        for model_kind in ["ERM", "DANN", "CORAL"]:
            for seed in CFG.SEEDS:
                try: real_rows.append(train_real_target(source, model_kind, seed))
                except Exception as exc: real_rows.append({"source": source, "model": model_kind, "seed": seed, "status": "FAILED", "error": repr(exc)})
    REAL_RESULTS = pd.DataFrame(real_rows)
    REAL_RESULTS.to_csv(OUT_ROOT / "tables" / "real_target_results.csv", index=False)
    display(REAL_RESULTS)
else:
    REAL_RESULTS = None
    print("Real-target training is disabled until the complete primary factorial is validated.")

## 13. Error analysis and reproducibility package

The final paper must show where each model fails, not only report one aggregate number. Error analysis is performed only after complete results exist and is stratified by topic, text length, and corpus. The notebook also writes a machine-readable manifest containing the configuration, split hashes, source summary, run keys, failures, and artifact paths.

In [ ]:
# 13.1 Error-analysis utilities

def error_table(y_true, prob_pos, frame):
    out = frame.copy().reset_index(drop=True)
    out["y_true"] = np.asarray(y_true).astype(int)
    out["prob_pos"] = np.asarray(prob_pos)
    out["y_pred"] = (out["prob_pos"] >= 0.5).astype(int)
    out["error_type"] = np.select([
        (out.y_true == 0) & (out.y_pred == 1),
        (out.y_true == 1) & (out.y_pred == 0),
    ], ["false_positive", "false_negative"], default="correct")
    return out

if FACTORIAL_RESULTS is not None:
    print("Error analysis is available after loading a selected checkpoint and generating test probabilities.")
    # Deliberately no aggregate claim is made from a single seed.

### 13.1b Topic-stratified accuracy

Using `reload_predictions` (Section 8.5) on the no-shift `(0,0)` cell for every completed model and seed, this cell aggregates accuracy by topic across the six annotated categories (political, religious, personal, geopolitical, gender-abusive, non-abusive), giving the fairness-relevant stratified breakdown.

In [ ]:
# 13.1b Topic-stratified accuracy, aggregated over completed models and seeds
if FACTORIAL_RESULTS is not None:
    topic_frames = []
    for model_kind in FACTORIAL_RESULTS["model"].unique():
        for seed in CFG.SEEDS:
            et = reload_predictions(model_kind, seed, 0, 0)
            if et is not None:
                topic_frames.append(et)
    if topic_frames:
        TOPIC_ERRORS = pd.concat(topic_frames, ignore_index=True)
        TOPIC_SUMMARY = (TOPIC_ERRORS.assign(correct=lambda d: d.error_type == "correct")
                          .groupby(["model", "topic"])["correct"].mean().rename("accuracy").reset_index())
        display(TOPIC_SUMMARY.pivot(index="topic", columns="model", values="accuracy"))
        TOPIC_SUMMARY.to_csv(OUT_ROOT / "tables" / "topic_stratified_accuracy.csv", index=False)
    else:
        TOPIC_SUMMARY = None
        print("No reloadable checkpoints found on disk yet for topic-stratified analysis.")
else:
    TOPIC_SUMMARY = None


In [ ]:
# 13.1c Figure -- topic-stratified accuracy heatmap
if TOPIC_SUMMARY is not None and len(TOPIC_SUMMARY):
    pivot = TOPIC_SUMMARY.pivot(index="topic", columns="model", values="accuracy")
    fig, ax = plt.subplots(figsize=TACL_2COL)
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="Blues", vmin=0.4, vmax=1.0,
                cbar_kws={"label": "Accuracy (no-shift cell)"}, linewidths=0.5, linecolor="white", ax=ax)
    ax.set_xlabel(""); ax.set_ylabel("")
    ax.set_title("Topic-stratified accuracy, no-shift cell", loc="left", fontsize=10)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "fig_topic_error_heatmap.pdf")
    fig.savefig(FIG_DIR / "fig_topic_error_heatmap.png")
    plt.show()
else:
    print("Topic-error heatmap is pending checkpoint availability.")


In [ ]:
# 13.2 Final completeness and artifact manifest

def final_manifest():
    files = []
    for p in OUT_ROOT.rglob("*"):
        if p.is_file():
            files.append({"path": str(p.relative_to(OUT_ROOT)), "bytes": p.stat().st_size, "sha256": hashlib.sha256(p.read_bytes()).hexdigest()[:16]})
    manifest = {"run_id": RUN_ID, "config": jsonable_config(), "files": files,
                "claims": {
                    "synthetic_content_protocol_attribution": "allowed only with complete factorial and pre-specified interventions",
                    "real_platform_language_causal_attribution": False,
                    "target_label_model_selection": False,
                }}
    (OUT_ROOT / "manifests" / "artifact_manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
    return manifest

manifest = final_manifest()
print("Artifact manifest written with", len(manifest["files"]), "files.")

## 14. Final paper-facing interpretation template

The following text is intentionally conservative and should be populated only from completed tables:

> We evaluated a two-factor intervention design on a disjoint Bangla anchor corpus. Content shift was operationalized as class-preserving topic-composition reweighting; protocol shift was operationalized as a pre-specified class-conditional transition from latent to observed training labels. Exact Shapley values decomposed the performance loss in the complete factorial. We additionally evaluated external corpora as natural stress tests, but did not assign them causal platform/language labels or transfer synthetic Shapley values without an out-of-distribution qualification.

This statement should not be strengthened beyond what the completed tables support; the design intentionally trades a broader four-factor claim for a narrower, falsifiable one.

In [ ]:
# 14.1 Fail-closed paper-facing claim gate
expected_n = len(EXPECTED_KEYS)
observed_n = 0 if FACTORIAL_RESULTS is None else len(FACTORIAL_RESULTS)
failures_path = OUT_ROOT / "tables" / "factorial_failures.csv"
if not failures_path.exists():
    no_known_failures = True
else:
    failure_df = pd.read_csv(failures_path)
    if "run_id" not in failure_df.columns:
        no_known_failures = False
    else:
        no_known_failures = len(failure_df[failure_df["run_id"].astype(str) == str(RUN_ID)]) == 0

CLAIM_GATE = {
    "complete_factorial": FACTORIAL_RESULTS is not None and observed_n == expected_n,
    "minimum_seeds": len(CFG.SEEDS) >= 5,
    "no_known_failures": no_known_failures,
    "external_causal_attribution_allowed": False,
    "target_labels_used_for_model_selection": False,
    "oracle_protocol_correction_must_be_labeled": "PROTOCOL_CORRECTED" in CFG.MODELS,
    "robustness_claim_allowed": ROBUSTNESS_RESULTS is not None and len(ROBUSTNESS_RESULTS) > 0,
}
if CLAIM_GATE["complete_factorial"]:
    assert observed_n == expected_n, "Unexpected duplicate or extra factorial rows."
print(json.dumps(CLAIM_GATE, indent=2, default=str))


## Execution status note

This notebook requires a complete current-run factorial before paper-facing claims are enabled. The primary factorial is 4 intervention cells × 6 models × 5 seeds = 120 runs. The oracle `PROTOCOL_CORRECTED` arm must be labelled as an oracle/upper-bound, not as a deployable method.


## 15. Submission checklist

Before submitting to TACL or ACL Rolling Review, complete the full factorial and robustness arm with no missing model–seed–cell combinations; report every failed run and all deviations from the pre-specified protocol; release the exact preprocessing, split manifest, configuration, checkpoint-selection rule, and analysis code; include confidence intervals and paired tests; compare against strong, recognized baselines; explain the limitations of the real-corpus stress test; and ensure that every paper claim is supported by a table or figure generated from the final artifact manifest.

Design and reporting discipline are necessary but not sufficient for acceptance; the completed results, the paper's novelty and clarity, and related-work positioning still determine the outcome.

## 16. References used in the protocol

[1] TACL submission guidelines: https://direct.mit.edu/tacl/pages/submission-guidelines

[2] ACL Rolling Review call for papers: http://aclrollingreview.org/cfp

[3] Ganin et al. (2016), *Domain-Adversarial Training of Neural Networks*: https://www.jmlr.org/papers/v17/15-239.html

[4] Kiryo et al. (2017), *Positive-Unlabeled Learning with Non-Negative Risk Estimator*: https://proceedings.neurips.cc/paper_files/paper/2017/hash/7cce53cf90577442771720a370c3c723-Abstract.html

[5] Mathew et al. (2021), *HateXplain: A Benchmark Dataset for Explainable Hate Speech Detection*: https://aaai.org/papers/14867-hatexplain-a-benchmark-dataset-for-explainable-hate-speech-detection/

[6] LREC 2022 BD-SHS resource paper: https://aclanthology.org/2022.lrec-1.552/

[7] NeurIPS paper checklist: https://neurips.cc/public/guides/PaperChecklist

[8] Annotation Matters: https://aclanthology.org/anthology-files/anthology-files/pdf/cas/2026.cas-1.10.pdf

[9] Blodgett et al. (2020), *Language (Technology) is Power: A Critical Survey of \"Bias\" in NLP*: https://aclanthology.org/2020.acl-main.485/